## Setup

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

from src.twin.model import HoloSystem

# Vector-safe export: all text stays live (editable) rather than outlined.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
mpl.rcParams['text.hinting'] = 'none'  # Disables the zero-rounding pixel snapping

# Mathtext forced onto a real installable font so Greek letters and operators
# stay live text too. No calligraphic glyphs, so labels use plain letters.
MATH_FONT = "DejaVu Sans"
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = MATH_FONT
mpl.rcParams["mathtext.it"] = f"{MATH_FONT}:italic"
mpl.rcParams["mathtext.bf"] = f"{MATH_FONT}:bold"
mpl.rcParams["mathtext.sf"] = MATH_FONT
mpl.rcParams["mathtext.cal"] = MATH_FONT
mpl.rcParams["mathtext.tt"] = "DejaVu Sans Mono"

# Shared figure style, used by every figure cell below.
FIG_WIDTH_MM = 180.0        # double-column width
FIG_DPI = 400
FONT_SIZE = 7
DEVICE = "cpu"

ACCENT_COLOR = "#FFCC01"    # ROI boxes, inset frames, pipeline arrow
SECONDARY_COLOR = "#0084FF"   # reference overlays, distinct from the accent
CMAP_RESIDUAL = "bwr"         # signed maps, symmetric about zero
CMAP_GRAY = "gray"          # intensities, captures, unwrapped phase
CMAP_AMP = "hot"            # amplitude of complex fields
CMAP_PHASE = "twilight"     # wrapped phase of complex fields (cyclic)

OUTPUT_DIR = Path("analysis/output")


def fig_path(name: str) -> Path:
    """Output path for a figure, creating the output directory on first use."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return OUTPUT_DIR / f"{name}.pdf"


def save_figure(fig, name: str, pad_inches: float = 0.0) -> Path:
    """Save a manually laid-out figure. The panel_layout builders save themselves."""
    path = fig_path(name)
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight", pad_inches=pad_inches)
    print(f"Saved: {path}")
    return path

## Data locations

In [ ]:
from src.config import DATA_ROOT as PAPER_ROOT

TARGETS_DIR = PAPER_ROOT / "targets"
CGH_DIR = PAPER_ROOT / "paper_cgh"
CROSSTALK_DIR = PAPER_ROOT / "paper_crosstalk_approach"
FIT_DIR = PAPER_ROOT / "paper_twin_fit"
REPEAT_DIR = PAPER_ROOT / "2026-08-13_twin_cgh"

# Trained twin checkpoints. ap / no_ap = physical SLM aperture fitted or not;
# 2pi / 4pi = modulation depth of the system the fit was run on.
CKPT_2PI_AP = FIT_DIR / "study_04_2pi_ap_fit/ap_2pi/background_stage/twin_model.pt"
CKPT_4PI_AP = FIT_DIR / "study_04_4pi_ap_fit/ap_4pi/background_stage/twin_model.pt"
CKPT_4PI_NO_AP = FIT_DIR / "study_04_4pi_ap_fit/no_ap_4pi/background_stage/twin_model.pt"
CKPT_4PI_AP_INIT = FIT_DIR / "study_04_4pi_ap_fit/ap_4pi/initial/twin_model.pt"
CKPT_4PI_AP_OPTICS = FIT_DIR / "study_04_4pi_ap_fit/ap_4pi/optics_stage/twin_model.pt"

BACKGROUND_MASK_PATH = FIT_DIR / "study_04_4pi_ap_fit/ap_4pi/background_mask/mask.pt"

## Layout

Panel-figure layout, inlined so the whole analysis lives in this notebook.

In [ ]:
"""
Reusable multi-row panel-figure layout: N image/text bands stacked
vertically, each split into evenly-sized columns (one per condition /
subfigure), with column and row gaps specified as absolute physical
sizes (mm) rather than GridSpec's fraction-of-average-cell convention.

Built for figures like "five ablation conditions x [letter, full image
with an ROI box, its metrics, the ROI crop, the ROI's metrics]", but the
row list is generic - any sequence of image/text bands works.

Panels accept a pre-cropped ROI image and a polygon outline, so a figure can
show a crop taken at native camera-pixel resolution while drawing its
(possibly rotated) footprint on a rectified full-field view.
"""
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Sequence

import math

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as patheffects
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.axes_size import Fixed


# =============================================================================
# ROI helpers
# =============================================================================

@dataclass(frozen=True)
class ROI:
    """Square region of interest in PIXEL coordinates (row0, col0 = top-left corner)."""
    row0: int
    col0: int
    size: int

    def slice(self) -> tuple:
        return (slice(self.row0, self.row0 + self.size),
                slice(self.col0, self.col0 + self.size))


def crop_roi(img, roi: ROI):
    """Extract the ROI's pixels from `img`. The same ROI (location AND
    size) is meant to be reused across every panel in a comparison, so
    this is always literal integer-pixel indexing - no resampling."""
    return img[roi.slice()]


def center_crop(arr, width: int, height: int = None):
    """
    Centered rectangular crop of `arr` - `width` columns, `height` rows
    (defaults to `width`, i.e. a centered square) around the array's own
    center. For DISPLAY only - e.g. a field/mask whose outer region is
    mostly noise and only the central window is informative - while the
    full, uncropped array stays untouched for anything that needs to
    compute on it (registration, fitting, metrics, ...).
    """
    height = width if height is None else height
    h, w = arr.shape[:2]
    r0 = max((h - height) // 2, 0)
    c0 = max((w - width) // 2, 0)
    return arr[r0:r0 + height, c0:c0 + width]


def draw_roi_box(ax, roi: ROI, color: str, linewidth: float = 1.2):
    """
    Outline where `roi` sits within an already-imshow'd full image on
    `ax`. Anchored at (col0 - 0.5, row0 - 0.5) to align with imshow's
    default pixel-boundary convention (origin='upper': pixel i spans data
    coords [i - 0.5, i + 0.5]).
    """
    ax.add_patch(Rectangle(
        (roi.col0 - 0.5, roi.row0 - 0.5), roi.size, roi.size,
        edgecolor=color, facecolor="none", linewidth=linewidth,
    ))


def draw_roi_box_slices(ax, row_sl: slice, col_sl: slice, color: str,
                          linewidth: float = 1.2):
    """
    Same as draw_roi_box, but for a (row_slice, col_slice) pair - e.g.
    from roi_in_other_grid - rather than a square ROI object. Needed
    when the SAME physical ROI has been scaled onto a differently-shaped
    (and not necessarily square) grid, so the box itself isn't
    necessarily square either.
    """
    ax.add_patch(Rectangle(
        (col_sl.start - 0.5, row_sl.start - 0.5),
        col_sl.stop - col_sl.start, row_sl.stop - row_sl.start,
        edgecolor=color, facecolor="none", linewidth=linewidth,
    ))


def draw_panel_letter(ax, letter: str, font_size: float = 9, title: str = None):
    """
    Panel letter inside an image axes' top-left corner, white with a black
    stroke so it stays legible over both bright and near-black content.

    `title` adds a short keyword label beside the letter, at normal weight and
    one size down, so the panel says what it is without a trip to the caption.
    Multi-line titles are fine: subsequent lines are indented to clear the
    letter.
    """
    ax.text(0.03, 0.97, letter, transform=ax.transAxes,
            fontsize=font_size + 1, fontweight="bold", color="white",
            ha="left", va="top",
            path_effects=[patheffects.withStroke(linewidth=1.5, foreground="black")])

    if title:
        ax.text(0.03, 0.97, "    " + title.replace("\n", "\n    "),
                transform=ax.transAxes, fontsize=font_size, color="white",
                ha="left", va="top", linespacing=1.25,
                path_effects=[patheffects.withStroke(linewidth=1.5, foreground="black")])


CBAR_THICKNESS_IN = 0.07   # fixed physical colorbar thickness (inches)
CBAR_PAD_IN = 0.05         # fixed physical gap between panel and its colorbar (inches)


def add_horizontal_colorbar(fig, ax, im, label: str = None, font_size: float = 9,
                              thickness_in: float = CBAR_THICKNESS_IN,
                              pad_in: float = CBAR_PAD_IN):
    """
    Horizontal colorbar directly underneath `ax`, fixed PHYSICAL thickness
    (inches) regardless of `ax`'s own size - so colorbars stay matched
    across panels of different sizes in the same figure, same as Fig 2's
    (b)/(c)/(f)/(g) treatment (ported from that figure's _add_colorbar).
    Top-anchors `ax` first so the image doesn't recenter into the space
    the colorbar carves out of the bottom (append_axes ignores a plain
    ax.set_anchor call - it has to go through the divider).
    """
    div = make_axes_locatable(ax)
    div.set_anchor("N")
    cax = div.append_axes("bottom", size=Fixed(thickness_in), pad=Fixed(pad_in))
    cbar = fig.colorbar(im, cax=cax, orientation="horizontal")
    cbar.ax.tick_params(labelsize=font_size - 1)
    if label:
        cbar.set_label(label, fontsize=font_size)
    return cbar


def draw_roi_outline(ax, outline, color: str, linewidth: float = 1.2):
    """Draw a (N, 2) array of (col, row) points as a closed outline - the
    footprint of a camera-grid ROI on a rectified view, where the region is a
    possibly rotated quadrilateral rather than an axis-aligned box."""
    ax.plot(np.append(outline[:, 0], outline[0, 0]),
            np.append(outline[:, 1], outline[0, 1]),
            color=color, lw=linewidth, solid_joinstyle="miter")


def draw_axes_border(ax, color: str, linewidth: float = 1.2):
    """Frame the full extent of `ax` - e.g. to mark an ROI-crop panel as
    'this is the zoomed-in region', in the same colour as its roi box."""
    ax.add_patch(Rectangle(
        (0, 0), 1, 1, transform=ax.transAxes,
        edgecolor=color, facecolor="none", linewidth=linewidth, clip_on=False,
    ))


# =============================================================================
# Metrics text formatting
# =============================================================================

DEFAULT_METRIC_FORMATS = {"NMSE": "{:.3f}", "PSNR": "{:.1f} dB", "SSIM": "{:.3f}"}

def format_metrics(m: dict, keys: Sequence[str] = ("NMSE", "PSNR"),
                    formats: dict = None, sep: str = ", ") -> str:
    """
    e.g. format_metrics(m) -> "NMSE 0.129, PSNR 13.6 dB". `keys` controls
    which metrics get shown and in what order, independently per row --
    e.g. show fewer/different metrics for the ROI row than the full-frame
    row without touching how the metrics themselves are computed.
    """
    formats = {**DEFAULT_METRIC_FORMATS, **(formats or {})}
    return sep.join(f"{k} {formats.get(k, '{:.3f}').format(m[k])}" for k in keys)


# =============================================================================
# mm <-> GridSpec spacing conversion
#
# GridSpec's wspace/hspace are a FRACTION OF MEAN CELL SIZE, not an
# absolute size - these back-solve a desired absolute gap (mm) into that
# convention. Works for rows or columns, equal or unequal cell sizes.
# =============================================================================

def mm_total_with_gaps(sizes_mm: Sequence[float], gap_mm: float) -> float:
    """Total extent (mm) of `sizes_mm` laid out with `gap_mm` between each pair."""
    return sum(sizes_mm) + (len(sizes_mm) - 1) * gap_mm


def mm_gap_to_spacing(sizes_mm: Sequence[float], gap_mm: float) -> float:
    """wspace/hspace value that produces an actual gap of `gap_mm` between
    adjacent GridSpec cells of the given sizes."""
    n = len(sizes_mm)
    if n <= 1 or gap_mm == 0:
        return 0.0
    return n * gap_mm / sum(sizes_mm)


# =============================================================================
# Row spec + figure builder
# =============================================================================

@dataclass
class Row:
    """
    One horizontal band of the figure, split into `n_panels` equal-width
    columns.

    kind="image": content[j] is a 2-D array, imshow'd with `cmap`
        (interpolation="nearest" always - these are real captures/crops,
        never smoothed/upsampled for display). roi + roi_color draws a
        location box on top of it; border_color frames the whole cell
        (e.g. for an ROI-crop panel). Any combination of roi/border is
        fine, including neither.
    kind="text": content[j] is a string, placed at `text_pos` (axes
        fraction) per `text_kwargs` (ha/va/fontsize/... - anything
        ax.text takes).
    """
    height_mm: float
    kind: str                       # "image" | "text"
    content: Sequence
    cmap: str = "gray"
    roi: Optional[ROI] = None
    roi_color: Optional[str] = None
    border_color: Optional[str] = None
    border_linewidth: float = 1.2
    text_pos: tuple = (0.5, 0.5)
    text_kwargs: dict = field(default_factory=lambda: dict(
        fontsize=8, color="black", ha="center", va="center"))


def build_panel_figure(
    rows: Sequence[Row],
    n_panels: int,
    fig_width_mm: float,
    out_path,
    col_gap_mm: float = 1.0,
    row_gap_mm: float = 0.0,
    dpi: int = 400,
):
    """
    Render `rows` stacked vertically, each split into `n_panels` equal
    columns spanning the full figure width edge-to-edge (no left/right
    margin), and save to `out_path`.

    Column width is derived (as large as possible) from fig_width_mm and
    col_gap_mm; row heights come directly from each Row.height_mm --
    every band gets a real, explicitly-sized axes, so nothing relies on
    text overflowing past the nominal figure bounds. Figure height is
    whatever the rows add up to, not fixed.
    """
    if not rows:
        raise ValueError("rows must be non-empty")

    col_mm = (fig_width_mm - (n_panels - 1) * col_gap_mm) / n_panels
    row_heights_mm = [r.height_mm for r in rows]
    fig_height_mm = mm_total_with_gaps(row_heights_mm, row_gap_mm)

    fig = plt.figure(figsize=(fig_width_mm / 25.4, fig_height_mm / 25.4))
    wspace = mm_gap_to_spacing([col_mm] * n_panels, col_gap_mm)
    hspace = mm_gap_to_spacing(row_heights_mm, row_gap_mm)

    gs = fig.add_gridspec(
        len(rows), n_panels,
        height_ratios=row_heights_mm, wspace=wspace, hspace=hspace,
        left=0, right=1, top=1, bottom=0,
    )

    for i, row in enumerate(rows):
        if len(row.content) != n_panels:
            raise ValueError(
                f"Row {i} ({row.kind}) has {len(row.content)} entries, "
                f"expected n_panels={n_panels}."
            )
        for j in range(n_panels):
            ax = fig.add_subplot(gs[i, j])
            ax.axis("off")

            if row.kind == "image":
                ax.imshow(row.content[j], cmap=row.cmap, interpolation="nearest")
                ax.set_box_aspect(1)
                if row.roi is not None and row.roi_color is not None:
                    draw_roi_box(ax, row.roi, row.roi_color)
                if row.border_color is not None:
                    draw_axes_border(ax, row.border_color, row.border_linewidth)

            elif row.kind == "text":
                ax.text(*row.text_pos, row.content[j], transform=ax.transAxes,
                         **row.text_kwargs)
            else:
                raise ValueError(f"Unknown row kind {row.kind!r}")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0)
    return fig


# =============================================================================
# Grid figure - N self-contained panels (letter / image+ROI+inset / metrics),
# wrapped into a fixed number of columns, EACH PANEL CARRYING ITS OWN ROI.
#
# build_panel_figure (above) is for "same condition list, one shared ROI,
# ROI shown as its own row below the full images" - e.g. an ablation
# sweep of the same scene. This is for the opposite case: visually
# different content per panel (different target types, different
# datasets...) where a single shared ROI location wouldn't make sense,
# and the inset is drawn INSIDE the image (corner overlay) rather than
# as a separate row, so panels can be bigger / fewer per row.
#
# Column/row placement is done by hand (explicit mm -> figure-fraction
# axes, via fig.add_axes) rather than through GridSpec, because GridSpec
# only has one uniform wspace/hspace per figure - there's no way to ask
# it for a small gap between a panel's own letter/image/metrics bands
# and a larger gap between separate panels. Explicit placement makes
# that distinction directly.
# =============================================================================

@dataclass
class Panel:
    """One self-contained grid panel: an image with its OWN roi (may
    differ in location and size from every other panel's), and its own
    pre-formatted metrics string (build with format_metrics - full-frame,
    ROI, or both combined, whatever the caller wants under this panel)."""
    image: object
    letter: str
    metrics_text: str
    roi: Optional[ROI] = None
    roi_color: Optional[str] = ACCENT_COLOR
    cmap: str = "gray"
    inset_image: object = None      # pre-cropped ROI; overrides crop_roi(image, roi)
    roi_outline: object = None      # (N, 2) polygon; overrides the axis-aligned box
    annotate: object = None         # annotate(ax, panel) called on the ROI inset
    annotate_image: object = None   # annotate(ax, panel) called on the full image
    target_inset: object = None     # reference crop, drawn beside the measured one


def axis_positions_mm(sizes_mm: Sequence[float], gaps_mm, total_mm: float) -> list:
    """
    Start/size (as fractions of `total_mm`) for `sizes_mm` laid out in
    order, edge-to-edge (no leading/trailing margin). `gaps_mm` is either
    one number (constant gap between every pair) or a list of length
    len(sizes_mm) - 1 (a different gap between each pair - e.g. a small
    gap within a panel's own bands and a larger gap between panels).
    """
    n = len(sizes_mm)
    if not isinstance(gaps_mm, (list, tuple)):
        gaps_mm = [gaps_mm] * max(n - 1, 0)
    elif len(gaps_mm) != n - 1:
        raise ValueError(f"gaps_mm must have length {n - 1}, got {len(gaps_mm)}")

    positions = []
    pos = 0.0
    for i, s in enumerate(sizes_mm):
        if i > 0:
            pos += gaps_mm[i - 1]
        positions.append((pos / total_mm, s / total_mm))
        pos += s
    return positions


def draw_image_inset(ax, inset_image, color: str, linewidth: float = 1.2,
                       cmap: str = "gray", width_frac: float = 0.35,
                       height_frac: float = 0.35, margin_frac: float = 0.03,
                       corner: str = "lower right", vmin=None, vmax=None):
    """
    Overlay `inset_image` (ANY 2-D array - not necessarily related to
    whatever `ax` already shows) small, in one corner of `ax`, framed in
    `color`. width_frac/height_frac/margin_frac are fractions of `ax`'s
    own size, not the figure's.

    General building block behind draw_inset (below), which is the
    special case where the inset is a crop of the SAME image already
    shown on `ax`. Use this one directly when the inset is a genuinely
    different image - e.g. a related field in another domain/plane, a
    reference frame, a diagram.
    """
    corners = {
        "lower right": (1 - margin_frac - width_frac, margin_frac),
        "lower left":  (margin_frac, margin_frac),
        "upper right": (1 - margin_frac - width_frac, 1 - margin_frac - height_frac),
        "upper left":  (margin_frac, 1 - margin_frac - height_frac),
    }
    # set_box_aspect(1) below shrinks the requested [x0,y0,w,h] box down to a
    # square around its ANCHOR point (default anchor is "C", the box's own
    # centre). When the parent `ax` is non-square, [x0,y0,w,h] - a fraction
    # of ax's rectangular bbox - is itself rectangular in display space, so
    # centring the square shrink drags the inset away from the true corner of
    # `ax` and toward the middle of that rectangle. Anchoring to the matching
    # compass point instead keeps that corner of the box fixed and shrinks
    # inward from there, so the inset stays flush against the real corner of
    # `ax` regardless of its aspect ratio.
    anchors = {
        "lower right": "SE",
        "lower left":  "SW",
        "upper right": "NE",
        "upper left":  "NW",
    }
    if corner not in corners:
        raise ValueError(f"corner must be one of {list(corners)}, got {corner!r}")
    x0, y0 = corners[corner]

    inset_ax = ax.inset_axes([x0, y0, width_frac, height_frac])
    inset_ax.set_anchor(anchors[corner])
    inset_ax.set_box_aspect(1)  # force the axes BOX itself square, not just the
                                  # data - width_frac/height_frac are fractions
                                  # of the parent's (possibly non-square) bbox, so
                                  # equal fractions alone don't guarantee a square
                                  # box; without this the border frame comes out
                                  # rectangular even when the sampled image is square
    # extent=(0,1,0,1) forces the image to FILL that square box exactly,
    # regardless of inset_image's own pixel aspect ratio - imshow's default
    # aspect="equal" would otherwise letterbox non-square data inside the
    # (now square) box, leaving empty margins rather than a full square image.
    inset_ax.imshow(inset_image, cmap=cmap, interpolation="nearest",
                      vmin=vmin, vmax=vmax, extent=(0, 1, 0, 1))
    inset_ax.axis("off")  # removes the default axes spines too - without this,
                            # the default black spine sits right at the same edge
                            # as draw_axes_border's patch below, showing through
                            # as a second, mismatched frame colour
    draw_axes_border(inset_ax, color, linewidth)
    return inset_ax


def draw_inset(ax, image, roi: ROI, color: str, linewidth: float = 1.2,
                cmap: str = "gray", width_frac: float = 0.35,
                height_frac: float = 0.35, margin_frac: float = 0.03,
                corner: str = "lower right"):
    """
    Overlay `crop_roi(image, roi)`, blown up, directly on top of `ax`
    (which has already imshow'd the full `image`) - in the given corner,
    framed in `color` (matching the ROI box drawn on the full image, so
    it visually reads as "this box, enlarged"). Thin wrapper around
    draw_image_inset for the common case where the inset IS a crop of
    the panel's own image.
    """
    return draw_image_inset(ax, crop_roi(image, roi), color, linewidth=linewidth,
                              cmap=cmap, width_frac=width_frac, height_frac=height_frac,
                              margin_frac=margin_frac, corner=corner)


def resize_to_square_nearest(arr, size: int = None):
    """
    Nearest-neighbour resize a 2-D array onto a `size` x `size` grid
    (default: max(arr.shape) - upsamples the shorter axis to match the
    longer one, never downsamples/crops). For aligning something defined
    on a different native grid (e.g. a mask) onto the same square pixel
    coordinates as the images it's shown/cropped alongside, so the same
    ROI can be applied to both.
    """
    from PIL import Image

    arr = np.asarray(arr)
    h, w = arr.shape
    size = size or max(h, w)
    resized = Image.fromarray(arr.astype(np.float32)).resize((size, size), Image.NEAREST)
    return np.asarray(resized).astype(arr.dtype, copy=False)


def roi_in_other_grid(roi: ROI, from_shape: tuple, to_shape: tuple) -> tuple:
    """
    Convert `roi` (pixel coordinates on a grid of `from_shape`) into the
    (row_slice, col_slice) selecting the SAME physical region on a grid
    of `to_shape` - for two arrays that span the identical field of view
    but are stored at different resolutions/aspect ratios (e.g. a model's
    native rectangular far-field grid vs. the square grid affine_inverse
    resamples camera captures onto). Scales each axis independently by
    that axis's own from/to ratio, so the result is a plain (possibly
    non-square) crop - ROI itself stays square-only by construction.
    """
    row_scale = to_shape[0] / from_shape[0]
    col_scale = to_shape[1] / from_shape[1]
    r0 = round(roi.row0 * row_scale)
    c0 = round(roi.col0 * col_scale)
    r1 = round((roi.row0 + roi.size) * row_scale)
    c1 = round((roi.col0 + roi.size) * col_scale)
    return slice(r0, r1), slice(c0, c1)


def overlay_binary_mask(ax, mask, color: str = "black", extent=(0, 1, 0, 1)):
    """
    Overlay `mask` (values in [0,1], same shape as whatever's already
    imshow'n on `ax`) as an RGBA layer directly on top - OPAQUE `color`
    where mask==0 (masked-out / excluded region), fully TRANSPARENT where
    mask==1 (region actually used), so the underlying image shows through
    exactly where the mask says it's valid. interpolation="none" - meant
    to sit pixel-for-pixel over already-unresampled data below it, not
    get its own independent resampling.
    """
    mask = np.clip(np.asarray(mask, dtype=float), 0.0, 1.0)
    r, g, b = mcolors.to_rgb(color)
    rgba = np.zeros(mask.shape + (4,), dtype=float)
    rgba[..., 0], rgba[..., 1], rgba[..., 2] = r, g, b
    rgba[..., 3] = 1.0 - mask
    ax.imshow(rgba, extent=extent, interpolation="none")


def build_grid_figure(
    panels: Sequence[Panel],
    n_cols: int,
    fig_width_mm: float,
    out_path,
    col_gap_mm: float = 2.0,
    row_gap_mm: float = 3.0,
    sub_gap_mm: float = 1.0,
    letter_row_mm: float = 5.0,
    metrics_row_mm: float = 4.0,
    letters_in_image: bool = False,
    inset_width_frac: float = 0.35,
    inset_height_frac: float = 0.35,
    inset_margin_frac: float = 0.03,
    inset_corner: str = "lower right",
    font_size: float = 9,
    dpi: int = 400,
):
    """
    Render `panels` (each with its own image + roi) wrapped into `n_cols`
    columns (as many rows as needed), no left/right/top/bottom margin,
    fixed total width `fig_width_mm`. Each panel is letter / image (ROI
    box + blown-up corner inset drawn on top of it) / metrics text.

    col_gap_mm / row_gap_mm: gap between panels, across / down.
    sub_gap_mm: small gap between a single panel's own letter/image/
        metrics bands (independent of row_gap_mm between panels).
    letters_in_image: draw the letter in the image's top-left corner, stroked
        for legibility, instead of in a band above it. Removes that band.
    """
    if letters_in_image:
        letter_row_mm = 0.0
    if not panels:
        raise ValueError("panels must be non-empty")

    n = len(panels)
    n_rows = math.ceil(n / n_cols)

    panel_w_mm = (fig_width_mm - (n_cols - 1) * col_gap_mm) / n_cols
    img_mm = panel_w_mm  # square panels

    col_pos = axis_positions_mm([panel_w_mm] * n_cols, col_gap_mm, fig_width_mm)

    # Vertical stack, top to bottom: n_rows blocks of [letter, image,
    # metrics], sub_gap_mm within a block, row_gap_mm between blocks.
    v_sizes, v_gaps = [], []
    for r in range(n_rows):
        v_sizes += [letter_row_mm, img_mm, metrics_row_mm]
        v_gaps += [sub_gap_mm, sub_gap_mm]        # letter->image, image->metrics
        if r < n_rows - 1:
            v_gaps.append(row_gap_mm)             # this block -> next block
    fig_height_mm = sum(v_sizes) + sum(v_gaps)
    v_pos_top_down = axis_positions_mm(v_sizes, v_gaps, fig_height_mm)

    fig = plt.figure(figsize=(fig_width_mm / 25.4, fig_height_mm / 25.4))

    for idx, panel in enumerate(panels):
        r, c = divmod(idx, n_cols)
        x0, w = col_pos[c]
        block = 3 * r  # [letter, image, metrics] indices for this row

        letter_top, letter_h = v_pos_top_down[block]
        img_top, img_h = v_pos_top_down[block + 1]
        metrics_top, metrics_h = v_pos_top_down[block + 2]

        if not letters_in_image:
            letter_ax = fig.add_axes([x0, 1 - letter_top - letter_h, w, letter_h])
            letter_ax.axis("off")
            letter_ax.text(0, 0, panel.letter, transform=letter_ax.transAxes,
                           fontsize=font_size + 1, fontweight="bold",
                           color="black", ha="left", va="bottom")

        img_ax = fig.add_axes([x0, 1 - img_top - img_h, w, img_h])
        img_ax.axis("off")
        img_ax.imshow(panel.image, cmap=panel.cmap, interpolation="nearest")
        img_ax.set_box_aspect(1)
        if letters_in_image and panel.letter:
            draw_panel_letter(img_ax, panel.letter, font_size)
        if panel.roi is not None or panel.inset_image is not None:
            if panel.roi_color is not None:
                if panel.roi_outline is not None:
                    draw_roi_outline(img_ax, panel.roi_outline, panel.roi_color)
                elif panel.roi is not None:
                    draw_roi_box(img_ax, panel.roi, panel.roi_color)
            inset = (panel.inset_image if panel.inset_image is not None
                     else crop_roi(panel.image, panel.roi))
            inset_ax = draw_image_inset(
                img_ax, inset, panel.roi_color, cmap=panel.cmap,
                width_frac=inset_width_frac, height_frac=inset_height_frac,
                margin_frac=inset_margin_frac, corner=inset_corner)
            if panel.annotate is not None:
                panel.annotate(inset_ax, panel)
        if panel.annotate_image is not None:
            panel.annotate_image(img_ax, panel)

        metrics_ax = fig.add_axes([x0, 1 - metrics_top - metrics_h, w, metrics_h])
        metrics_ax.axis("off")
        metrics_ax.text(0.5, 0.5, panel.metrics_text, transform=metrics_ax.transAxes,
                         fontsize=font_size - 1, color="black", ha="center", va="center")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0)
    return fig

In [ ]:
# =============================================================================
# Multi-row panel grid - an n_rows x n_cols GRID of Fig 3/5-style condition
# blocks (full image w/ ROI box / its metrics / the SAME ROI crop, blown up
# to panel size / that crop's own metrics), sharing ONE roi across every
# panel. build_panel_figure (above) does exactly this block but only for a
# SINGLE row of n_panels; this generalizes it to an arbitrary grid so two
# single-row comparison figures can be merged into one multi-row one
# without giving up the "full image, then its own ROI crop below it"
# layout build_panel_figure/Row already do well. build_grid_figure (above)
# looks similar but is for the opposite case - visually different content
# per panel with its OWN roi, inset drawn as a small corner overlay rather
# than a same-size row underneath.
# =============================================================================

@dataclass
class GridPanel:
    """One cell of a build_multi_row_panel_figure grid. metrics_text /
    roi_metrics_text left as "" just leaves that band blank - e.g. for a
    reference image that isn't scored - rather than omitting the band, so
    every column stays pixel-aligned regardless of which cells have metrics."""
    image: object
    metrics_text: str = ""
    roi_metrics_text: str = ""
    cmap: str = "gray"
    letter: str = ""   # one letter per GROUP, drawn on the full-image panel only
    title: str = ""    # short keyword label drawn beside the letter
    roi_image: object = None    # pre-cropped ROI; overrides crop_roi(image, roi)
    roi_outline: object = None  # (N, 2) polygon; overrides the axis-aligned box
    annotate: object = None     # annotate(ax, panel) called on the ROI crop


def build_multi_row_panel_figure(
    grid: Sequence[Sequence[Optional[GridPanel]]],
    fig_width_mm: float,
    out_path=None,
    roi: Optional[ROI] = None,
    roi_color: Optional[str] = ACCENT_COLOR,
    col_gap_mm: float = 1.0,
    row_gap_mm: float = 3.0,
    sub_gap_mm: float = 0.5,
    metrics_row_mm: float = 3.5,
    extra_bottom_mm: float = 0.0,
    font_size: float = 8,
    dpi: int = 400,
):
    """
    Render `grid` (a list of rows of GridPanel | None, all rows the same
    length) as a grid of comparison blocks, each stacked top to bottom as:
    full image (with `roi` boxed on it) / its metrics_text / the same `roi`
    crop blown up to panel size / that crop's roi_metrics_text. A None cell is
    left blank. Panels are square, splitting fig_width_mm across n_cols.

    extra_bottom_mm reserves empty height below the grid for the caller to add
    its own axes. out_path=None builds without saving.

    Returns the Figure.
    """
    if not grid or not grid[0]:
        raise ValueError("grid must be non-empty")
    n_rows = len(grid)
    n_cols = len(grid[0])
    for r, row in enumerate(grid):
        if len(row) != n_cols:
            raise ValueError(f"grid row {r} has {len(row)} entries, expected {n_cols}")

    panel_mm = (fig_width_mm - (n_cols - 1) * col_gap_mm) / n_cols   # square panels
    col_pos = axis_positions_mm([panel_mm] * n_cols, col_gap_mm, fig_width_mm)

    # Vertical stack, top to bottom: n_rows blocks of [image, metrics,
    # roi-crop, roi-metrics], sub_gap_mm within a block, row_gap_mm between.
    v_sizes, v_gaps = [], []
    for r in range(n_rows):
        v_sizes += [panel_mm, metrics_row_mm, panel_mm, metrics_row_mm]
        v_gaps += [sub_gap_mm, sub_gap_mm, sub_gap_mm]
        if r < n_rows - 1:
            v_gaps.append(row_gap_mm)
    grid_height_mm = sum(v_sizes) + sum(v_gaps)
    fig_height_mm = grid_height_mm + extra_bottom_mm

    # v_pos is computed against the GRID height, then rescaled onto the taller
    # figure, so the reserved band sits below an otherwise unchanged grid.
    v_pos = axis_positions_mm(v_sizes, v_gaps, grid_height_mm)
    scale = grid_height_mm / fig_height_mm
    v_pos = [(top * scale, size * scale) for top, size in v_pos]

    fig = plt.figure(figsize=(fig_width_mm / 25.4, fig_height_mm / 25.4))

    for r in range(n_rows):
        block = 4 * r
        img_top, img_h = v_pos[block]
        metrics_top, metrics_h = v_pos[block + 1]
        inset_top, inset_h = v_pos[block + 2]
        inset_metrics_top, inset_metrics_h = v_pos[block + 3]

        for c in range(n_cols):
            panel = grid[r][c]
            if panel is None:
                continue
            x0, w = col_pos[c]

            img_ax = fig.add_axes([x0, 1 - img_top - img_h, w, img_h])
            img_ax.axis("off")
            img_ax.imshow(panel.image, cmap=panel.cmap, interpolation="nearest")
            img_ax.set_box_aspect(1)
            if roi_color is not None:
                if panel.roi_outline is not None:
                    draw_roi_outline(img_ax, panel.roi_outline, roi_color)
                elif roi is not None:
                    draw_roi_box(img_ax, roi, roi_color)
            if panel.letter:
                draw_panel_letter(img_ax, panel.letter, font_size, panel.title)

            metrics_ax = fig.add_axes([x0, 1 - metrics_top - metrics_h, w, metrics_h])
            metrics_ax.axis("off")
            if panel.metrics_text:
                metrics_ax.text(0.5, 0.5, panel.metrics_text, transform=metrics_ax.transAxes,
                                fontsize=font_size, color="black", ha="center", va="center")

            if roi is not None or panel.roi_image is not None:
                inset_ax = fig.add_axes([x0, 1 - inset_top - inset_h, w, inset_h])
                inset_ax.axis("off")
                inset_ax.imshow(panel.roi_image if panel.roi_image is not None
                                else crop_roi(panel.image, roi),
                                cmap=panel.cmap, interpolation="nearest")
                inset_ax.set_box_aspect(1)
                if roi_color is not None:
                    draw_axes_border(inset_ax, roi_color)
                if panel.annotate is not None:
                    panel.annotate(inset_ax, panel)

                inset_metrics_ax = fig.add_axes(
                    [x0, 1 - inset_metrics_top - inset_metrics_h, w, inset_metrics_h])
                inset_metrics_ax.axis("off")
                if panel.roi_metrics_text:
                    inset_metrics_ax.text(0.5, 0.5, panel.roi_metrics_text,
                                          transform=inset_metrics_ax.transAxes,
                                          fontsize=font_size, color="black",
                                          ha="center", va="center")

    if out_path is not None:
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0)

    return fig

## Shared helpers

Capture loading, grid transforms, registration, validity masks and metrics.
Scoring happens on the CAMERA grid: the measurement is never resampled, and
the target crosses the fitted camera model exactly once.

In [ ]:
"""
Camera-space capture analysis.

Every metric is computed on the camera grid. The measurement is the fixed
point: it is never warped, never rescaled, never shifted. The target is
brought to it by passing through the twin's own camera model, and the
prediction is produced there directly.

Registration is one offset per acquisition group, estimated in target space
and folded into the camera model's affine shift parameter, so the target and
the prediction inherit it through the same warp rather than being resampled a
second time.

Scoring covers the first-order field of view, as a quadrilateral mapped from
the far-field corners, minus the central region the CGH excluded, eroded for
the interpolation kernel's reach.

Two display views exist and neither is ever scored: the rectified full field
(camera -> far field, for legibility) and a camera-pixel crop of the field of
view (native sensor resolution, for ROIs).
"""
import warnings
from contextlib import contextmanager
from dataclasses import dataclass

import numpy as np
import torch
from PIL import Image
from matplotlib.path import Path as MplPath
from skimage.metrics import structural_similarity as ssim_fn
from skimage.registration import phase_cross_correlation
from skimage.measure import find_contours
from scipy.ndimage import shift as nd_shift, binary_erosion

from src.loss.masked_ssim import MaskedSSIM

MAX_REGISTRATION_SHIFT_PX = 8.0
REGISTRATION_UPSAMPLE = 20
AMBIGUITY_RATIO = 0.99
AMBIGUITY_EXCLUSION_PX = 4
MASK_EROSION_PX = 2
ZOD_RADIUS_FACTOR = 20.0          # matches generate_holograms' centre mask
BACKGROUND_PERCENTILE = 10
SATURATION_LEVEL = 65535.0
SATURATION_MARGIN = 0.995
SHIFT_SIGN = -1.0                 # flip if folding a shift increases the residual


def to_numpy(x) -> np.ndarray:
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)


def _tensor(a) -> torch.Tensor:
    return torch.from_numpy(np.asarray(a, dtype=np.float64)).float().to(DEVICE)


# ---------------------------------------------------------------------------
# Loading
# ---------------------------------------------------------------------------

def load_capture_mean(base_dir, batch_size: int = 4, n_batches: int = None) -> np.ndarray:
    """
    Mean capture on the camera's raw pixel grid.

    n_batches is None : base_dir holds {k}.npy directly.
    n_batches is set  : base_dir holds batch_{n}/captures/{k}.npy; each batch is
                        averaged over its own captures, then the batch means are
                        averaged together.
    """
    if n_batches is None:
        frames = [np.load(Path(base_dir, f"{k}.npy")) for k in range(batch_size)]
        return sum(frames) / batch_size

    batch_means = []
    for n in range(n_batches):
        frames = [np.load(Path(base_dir, f"batch_{n}", "captures", f"{k}.npy"))
                  for k in range(batch_size)]
        batch_means.append(sum(frames) / batch_size)
    return sum(batch_means) / n_batches


def load_holograms(base_dir, n_batches: int = None) -> np.ndarray:
    """Holograms for a condition, stacked over batches to match the capture average."""
    if n_batches is None:
        return np.load(Path(base_dir, "holograms.npy"))
    return np.concatenate(
        [np.load(Path(base_dir, f"batch_{n}", "holograms.npy")) for n in range(n_batches)],
        axis=0)


def load_target_image(path, shape) -> np.ndarray:
    """Grayscale target resized onto `shape`, normalised to its own pre-resize max."""
    shape = tuple(shape)
    t = Image.open(path).convert("L")
    raw_max = np.asarray(t, dtype=np.float64).max()
    if t.size[::-1] != shape:
        t = t.resize((shape[1], shape[0]), Image.BILINEAR)
    return np.asarray(t, dtype=np.float64) / raw_max


def load_target_far(model: HoloSystem, path) -> np.ndarray:
    """Target intensity on the model's far-field grid."""
    return load_target_image(path, model.geometry.far_fov_samples)


def assert_first_order(model: HoloSystem, tag: str = "") -> None:
    """The rectified view matches the far-field grid only when fov == 1."""
    assert model.geometry.fov == 1.0, (
        f"{tag} geometry.fov != 1.0: the rectified view covers more than the "
        "first order.")


# ---------------------------------------------------------------------------
# Affine
#
# affine_params["shift"] is in camera pixels, injected by _compute_matrix as a
# standalone translation column built only from static constants. A far-field
# pixel offset therefore converts to a shift delta by a pure scalar, with no
# coupling to the learnable rotation or scale.
# ---------------------------------------------------------------------------

def base_scale(model: HoloSystem) -> tuple:
    g = model.geometry
    Hc, Wc = g.camera_shape
    fov_phys = g.focal_length * g.wavelength / g.slm_pixel_pitch * g.fov
    return (Wc * g.camera_pixel_pitch / fov_phys,
            Hc * g.camera_pixel_pitch / fov_phys)


def far_shift_to_camera_shift(model: HoloSystem, dy_far: float, dx_far: float) -> tuple:
    """Far-field pixel offset (dy, dx) -> affine shift delta in camera pixels (x, y)."""
    g = model.geometry
    Hc, Wc = g.camera_shape
    Mf, Nf = g.far_fov_samples
    base_sx, base_sy = base_scale(model)
    return (dx_far * Wc / (Nf * base_sx), dy_far * Hc / (Mf * base_sy))


@contextmanager # to use with "with" statements
def registered_camera(model: HoloSystem, registration):
    """Forward passes with a registration offset folded into the affine shift."""
    if registration is None or (registration.dy == 0.0 and registration.dx == 0.0):
        yield
        return

    original = model.camera.affine_params
    dsx, dsy = far_shift_to_camera_shift(model, registration.dy, registration.dx)
    try:
        model.camera.affine_params = {
            "rotation": original["rotation"],
            "scale": original["scale"],
            "shift": [original["shift"][0] + SHIFT_SIGN * dsx,
                      original["shift"][1] + SHIFT_SIGN * dsy],
        }
        yield
    finally:
        model.camera.affine_params = original


def render_to_camera(model: HoloSystem, far_field: np.ndarray, camera_shape: tuple,
                     saturate: bool = True) -> np.ndarray:
    """
    Far-field intensity -> camera grid, through the model's own supersampled
    pixel integration and saturation clipping.
    """
    with torch.no_grad():
        I = _tensor(far_field).unsqueeze(0)
        C = model.camera._affine(I, camera_shape=tuple(camera_shape))
        if saturate:
            C = model.camera._saturate(C)
    return to_numpy(C.squeeze(0))


def rectify_to_far_field(img: np.ndarray, model: HoloSystem,
                         normalize: bool = True, square: bool = False) -> np.ndarray:
    """
    Camera grid -> far-field grid, for display and for estimating a shift.

    The far field is sampled anisotropically - Mf x Nf samples over a square
    angular extent - so the native grid renders squashed. square=True resamples
    onto max(Mf, Nf)^2 instead, which is the right proportions for display and
    the wrong ones for measuring an offset: a shift recovered there is not in
    native far-field pixels and would not convert to an affine shift correctly.
    Display passes True; the registration path passes False.

    Interpolates whatever it is given, so it never touches an array that gets
    scored as an image.
    """
    raw_max = img.max()
    with torch.no_grad():
        out = model.camera.affine_inverse(_tensor(img).unsqueeze(0), square=square)
    out = to_numpy(out.squeeze(0))
    return out / raw_max if normalize else out


# ---------------------------------------------------------------------------
# Field of view in camera space
# ---------------------------------------------------------------------------

def _effective_matrix(model: HoloSystem, camera_shape: tuple) -> np.ndarray:
    """3x3 camera-normalised -> far-field-normalised, for this capture size."""
    Htrain, Wtrain = model.geometry.camera_shape
    Hfull, Wfull = camera_shape
    S = np.diag([Wfull / Wtrain, Hfull / Htrain, 1.0])
    return to_numpy(model.camera.matrix).astype(np.float64) @ S


def fov_polygon_camera(model: HoloSystem, camera_shape: tuple) -> np.ndarray:
    """(4, 2) camera-pixel (x, y) corners of the first-order field of view."""
    Hc, Wc = camera_shape
    edge = 1.0 / model.geometry.fov
    corners = np.array([[-edge, -edge, 1.0], [edge, -edge, 1.0],
                        [edge, edge, 1.0], [-edge, edge, 1.0]]).T
    cam_norm = np.linalg.inv(_effective_matrix(model, camera_shape)) @ corners
    cam_norm = cam_norm[:2] / cam_norm[2]
    return np.stack([((cam_norm[0] + 1.0) * Wc - 1.0) / 2.0,
                     ((cam_norm[1] + 1.0) * Hc - 1.0) / 2.0], axis=1)


def square_for_display(img: np.ndarray, model: HoloSystem) -> np.ndarray:
    """
    Far-field array resampled to max(Mf, Nf)^2, for display.

    The far field carries Mf x Nf samples over a square angular extent, so the
    native array renders squashed. Nothing scored passes through here.
    """
    side = int(max(model.geometry.far_fov_samples))
    if tuple(img.shape[:2]) == (side, side):
        return img
    return np.asarray(
        Image.fromarray(np.asarray(img, dtype=np.float32)).resize(
            (side, side), Image.BILINEAR), dtype=np.float64)


def fov_mask(model: HoloSystem, camera_shape: tuple) -> np.ndarray:
    """Camera pixels inside the first-order field of view."""
    Hc, Wc = camera_shape
    ys, xs = np.mgrid[0:Hc, 0:Wc]
    points = np.stack([xs.ravel(), ys.ravel()], axis=1)
    inside = MplPath(fov_polygon_camera(model, camera_shape)).contains_points(points)
    return inside.reshape(Hc, Wc)


def zod_mask_camera(model: HoloSystem, camera_shape: tuple) -> np.ndarray:
    """
    Camera pixels covered by the central region the CGH excluded.

    The ellipse is rebuilt on the far-field grid with the same radius and
    aspect generate_holograms uses, then rendered through the camera model, so
    the excluded region is by construction the region no hologram attempted to
    control.
    """
    g = model.geometry
    Mf, Nf = g.far_fov_samples
    radius = ZOD_RADIUS_FACTOR * g.M / g.P
    aspect = g.Mf / g.Nf
    Y, X = np.mgrid[0:Mf, 0:Nf]
    disc = (((Y - Mf // 2) ** 2 + ((X - Nf // 2) * aspect) ** 2)
            < radius ** 2).astype(np.float64)
    return render_to_camera(model, disc, camera_shape, saturate=False) > 0.5


def metric_mask(model: HoloSystem, camera_shape: tuple) -> np.ndarray:
    """
    Valid scoring region: inside the field of view, outside the zero order,
    eroded for the interpolation kernel's reach.

    The registration offset is folded into the affine before the reference is
    rendered, so no array is shifted afterwards and no band of fabricated
    pixels appears; the erosion covers the resampling kernel alone.
    """
    mask = fov_mask(model, camera_shape) & ~zod_mask_camera(model, camera_shape)
    return (binary_erosion(mask, iterations=MASK_EROSION_PX)
            if MASK_EROSION_PX > 0 else mask)


def fov_bbox(model: HoloSystem, camera_shape: tuple) -> tuple:
    """(row_slice, col_slice) bounding the field of view, for display crops."""
    poly = fov_polygon_camera(model, camera_shape)
    Hc, Wc = camera_shape
    r0, r1 = int(np.floor(poly[:, 1].min())), int(np.ceil(poly[:, 1].max()))
    c0, c1 = int(np.floor(poly[:, 0].min())), int(np.ceil(poly[:, 0].max()))
    return (slice(max(r0, 0), min(r1 + 1, Hc)), slice(max(c0, 0), min(c1 + 1, Wc)))


def crop_to_fov(img: np.ndarray, model: HoloSystem) -> np.ndarray:
    """Crop a camera-grid image to the field of view without resampling."""
    row_sl, col_sl = fov_bbox(model, img.shape[:2])
    return img[row_sl, col_sl]


def describe_camera_grid(model: HoloSystem, camera_shape: tuple, label: str = "") -> None:
    """How the field of view lands on this capture: footprint, coverage, zero order."""
    poly = fov_polygon_camera(model, camera_shape)
    inside = fov_mask(model, camera_shape)
    zod = zod_mask_camera(model, camera_shape)
    scored = metric_mask(model, camera_shape)
    w = poly[:, 0].max() - poly[:, 0].min()
    h = poly[:, 1].max() - poly[:, 1].min()
    print(f"{label}capture {tuple(camera_shape)}, model grid {tuple(model.geometry.camera_shape)}")
    print(f"  FoV spans {h:.0f} x {w:.0f} px, {100 * inside.mean():.1f}% of the frame")
    print(f"  zero order {100 * zod.mean():.2f}%, scored after erosion "
          f"{100 * scored.mean():.1f}%")


# ---------------------------------------------------------------------------
# Registration
# ---------------------------------------------------------------------------

@dataclass
class Registration:
    """(dy, dx) in far-field pixels aligning a reference onto a measurement."""
    dy: float
    dx: float
    ambiguity: float
    ok: bool

    def magnitude(self) -> float:
        return float(np.hypot(self.dy, self.dx))

    def __str__(self) -> str:
        return (f"(dy={self.dy:+.2f}, dx={self.dx:+.2f}, amb={self.ambiguity:.2f}"
                f"{'' if self.ok else ', UNRELIABLE'})")


ZERO_REGISTRATION = Registration(0.0, 0.0, 0.0, True)


def estimate_shift(img: np.ndarray, ref: np.ndarray,
                   max_shift_px: float = MAX_REGISTRATION_SHIFT_PX) -> Registration:
    """
    (dy, dx) such that shifting `img` by it aligns it onto `ref`.

    The integer peak comes from an FFT cross-correlation searched only within
    max_shift_px, so a periodic target cannot lock onto a distant lattice peak
    and report it as a wrapped-around shift of hundreds of pixels. Sub-pixel
    refinement then runs on the pre-aligned pair, where the residual is under
    a pixel.

    The ambiguity ratio compares the best peak against the best value outside
    a radius wide enough to clear the peak's own width. It is a heuristic;
    agreement of the offset across conditions is the stronger evidence.
    """
    a = np.asarray(ref, dtype=np.float64)
    b = np.asarray(img, dtype=np.float64)
    corr = np.fft.fftshift(np.fft.irfft2(
        np.fft.rfft2(a - a.mean()) * np.conj(np.fft.rfft2(b - b.mean())), s=a.shape))

    cy, cx = a.shape[0] // 2, a.shape[1] // 2
    R = int(np.ceil(max_shift_px))
    win = corr[cy - R:cy + R + 1, cx - R:cx + R + 1]
    iy, ix = np.unravel_index(np.argmax(win), win.shape)
    peak = float(win[iy, ix])

    E = AMBIGUITY_EXCLUSION_PX
    runner = win.copy()
    runner[max(iy - E, 0):iy + E + 1, max(ix - E, 0):ix + E + 1] = -np.inf
    ambiguity = (float(runner.max() / peak)
                 if peak > 0 and np.isfinite(runner).any() else np.inf)

    cy0, cx0 = int(iy - R), int(ix - R)
    pre = nd_shift(b, shift=(cy0, cx0), order=3, mode="nearest")
    fine, _err, _phase = phase_cross_correlation(
        a, pre, upsample_factor=REGISTRATION_UPSAMPLE, normalization=None)

    # The refinement runs on an already-aligned pair, so its own output must be
    # sub-pixel. Anything larger means it locked onto something else - which a
    # measurement bearing no resemblance to its target invites - and the
    # bounded integer peak is kept instead.
    fdy, fdx = float(fine[0]), float(fine[1])
    refined = max(abs(fdy), abs(fdx)) <= 1.0
    if not refined:
        fdy = fdx = 0.0

    dy, dx = cy0 + fdy, cx0 + fdx
    ok = (refined and ambiguity < AMBIGUITY_RATIO
          and np.hypot(dy, dx) <= max_shift_px + 1.0)
    return Registration(dy=dy, dx=dx, ambiguity=ambiguity, ok=ok)


def estimate_target_shift(model: HoloSystem, measured: np.ndarray,
                          target_far: np.ndarray) -> Registration:
    """
    Offset between a capture and its target, measured in far-field pixels.

    The capture is rectified for this purpose only: the output is a scalar
    pair, not an image that gets scored. Estimating here rather than in camera
    space keeps the comparison off the coarser sensor sampling.
    """
    return estimate_shift(target_far, rectify_to_far_field(measured, model))


# ---------------------------------------------------------------------------
# Scoring
# ---------------------------------------------------------------------------

def fit_scale(img: np.ndarray, reference: np.ndarray, mask: np.ndarray) -> float:
    """a* = argmin_a ||img - a*reference||^2 over the valid region."""
    i, r = img[mask].astype(np.float64), reference[mask].astype(np.float64)
    denom = np.mean(r * r)
    return float(np.mean(i * r) / denom) if denom > 0 else 1.0


def masked_ssim(img: np.ndarray, reference: np.ndarray, mask: np.ndarray,
                data_range: float) -> float:
    """SSIM with invalid pixels excluded from every local window."""
    def t(a):
        return torch.from_numpy(np.asarray(a, dtype=np.float64)).float()[None, None]

    try:
        return float(MaskedSSIM(mask=t(mask.astype(np.float64)),
                                data_range=data_range)(t(img), t(reference)).item())
    except Exception as exc:                                    # noqa: BLE001
        warnings.warn(f"MaskedSSIM unavailable ({exc}); using the mask bounding box")
        rows, cols = np.where(mask)
        sl = (slice(rows.min(), rows.max() + 1), slice(cols.min(), cols.max() + 1))
        return float(ssim_fn(reference[sl], img[sl], data_range=data_range))


def compute_metrics(img: np.ndarray, reference: np.ndarray,
                    mask: np.ndarray = None) -> dict:
    """
    NMSE / PSNR / SSIM / Pearson r over the valid region, with the fraction of
    the frame that region covers.

    `reference` is scaled to `img`, never the reverse, so a measurement passed
    as `img` is never rescaled to flatter itself.
    """
    img = np.asarray(img, dtype=np.float64)
    reference = np.asarray(reference, dtype=np.float64)
    if mask is None:
        mask = np.ones(img.shape, dtype=bool)
    if not mask.any():
        raise ValueError("mask is empty - nothing to score")

    ref = fit_scale(img, reference, mask) * reference
    iv, rv = img[mask], ref[mask]
    mse = float(np.mean((iv - rv) ** 2))
    data_range = float(max(iv.max(), rv.max()) - min(iv.min(), rv.min()))

    a, b = iv - iv.mean(), rv - rv.mean()
    denom = np.sqrt(np.sum(a * a) * np.sum(b * b))

    return {
        "NMSE": mse / float(np.mean(rv ** 2)),
        "PSNR": float("inf") if mse == 0 else float(10 * np.log10(data_range ** 2 / mse)),
        "SSIM": masked_ssim(img, ref, mask, data_range),
        "r": float(np.sum(a * b) / denom) if denom > 0 else float("nan"),
        "valid": float(mask.mean()),
    }

# Two lines of metrics for the band beneath an image panel.
def format_full_and_roi(full_metrics, roi_metrics):
    return (f"Full: PSNR {full_metrics['PSNR']:.1f} dB, "
            f"SSIM {full_metrics['SSIM']:.3f}\n"
            f"ROI: PSNR {roi_metrics['PSNR']:.1f} dB, "
            f"SSIM {roi_metrics['SSIM']:.3f}")


# One line of metrics for a pair with no region of its own.
def format_metrics_line(metrics):
    return (f"PSNR {metrics['PSNR']:.1f} dB, "
            f"SSIM {metrics['SSIM']:.3f}")



def background_residual(img: np.ndarray, reference: np.ndarray, mask: np.ndarray) -> float:
    """Mean (img - reference) over the darkest decile of valid reference pixels."""
    ref = fit_scale(img, reference, mask) * reference
    iv, rv = img[mask], ref[mask]
    dark = rv <= np.percentile(rv, BACKGROUND_PERCENTILE)
    return float(np.mean(iv[dark] - rv[dark]))


# ---------------------------------------------------------------------------
# ROIs
# ---------------------------------------------------------------------------

def roi_from_fov(model: HoloSystem, camera_shape: tuple,
                 fy: float, fx: float, size_frac: float):
    """
    An ROI placed by its position within the field of view rather than by
    absolute pixel.

    Captures of different sizes put the same physical region at different
    pixel coordinates, so a fixed box cannot serve them all. fy, fx are
    fractions of the field of view (0.5, 0.5 is its centre) and size_frac is
    the box's width as a fraction of the field's shorter side.
    """
    row_sl, col_sl = fov_bbox(model, camera_shape)
    h, w = row_sl.stop - row_sl.start, col_sl.stop - col_sl.start
    size = max(int(round(size_frac * min(h, w))), 4)
    r0 = int(round(row_sl.start + fy * h - size / 2))
    c0 = int(round(col_sl.start + fx * w - size / 2))
    r0 = int(np.clip(r0, row_sl.start, row_sl.stop - size))
    c0 = int(np.clip(c0, col_sl.start, col_sl.stop - size))
    return ROI(row0=r0, col0=c0, size=size)


def validate_roi(roi, shape: tuple, label: str = "ROI") -> None:
    if (roi.row0 < 0 or roi.col0 < 0
            or roi.row0 + roi.size > shape[0] or roi.col0 + roi.size > shape[1]):
        raise ValueError(
            f"{label} rows {roi.row0}-{roi.row0 + roi.size}, cols {roi.col0}-"
            f"{roi.col0 + roi.size} does not fit a {tuple(shape)} frame. ROIs are "
            "in camera pixels on the capture's own grid.")


def camera_roi_outline(roi, model: HoloSystem, camera_shape: tuple) -> np.ndarray:
    """(N, 2) (col, row) outline of a camera-grid ROI on the rectified view."""
    validate_roi(roi, camera_shape, "camera ROI")
    stamp = np.zeros(camera_shape, dtype=np.float64)
    stamp[roi.slice()] = 1.0
    contours = find_contours(
        rectify_to_far_field(stamp, model, normalize=False, square=True), 0.5)
    if not contours:
        raise ValueError(
            f"ROI rows {roi.row0}-{roi.row0 + roi.size}, cols {roi.col0}-"
            f"{roi.col0 + roi.size} maps to nothing on the rectified grid - it lies "
            "outside the first order.")
    contour = max(contours, key=len)
    return np.stack([contour[:, 1], contour[:, 0]], axis=1)


def roi_extent_um(roi, model: HoloSystem) -> float:
    """Width of a camera-pixel ROI in micrometres at the sensor."""
    return roi.size * model.geometry.camera_pixel_pitch


def fov_extent_um(model: HoloSystem, camera_shape: tuple) -> float:
    """Width of the whole field of view, in micrometres at the sensor."""
    _row_sl, col_sl = fov_bbox(model, camera_shape)
    return (col_sl.stop - col_sl.start) * model.geometry.camera_pixel_pitch


def draw_scale_bar_extent(ax, extent_um: float, color: str = ACCENT_COLOR,
                          fraction: float = 0.3, font_size: float = None,
                          corner: str = "lower right") -> None:
    """
    Scale bar for an axes showing a region `extent_um` micrometres wide,
    rounded to a 1/2/5 length.

    Defined by the physical extent rather than a pixel count, so it is correct
    on any grid: a far-field array and a camera crop of the same region carry
    different sample counts but the same width.
    """
    font_size = FONT_SIZE - 1 if font_size is None else font_size
    raw = extent_um * fraction
    exp = 10 ** np.floor(np.log10(raw))
    length_um = float(min([1, 2, 5, 10], key=lambda m: abs(m * exp - raw)) * exp)

    frac = length_um / extent_um
    y = 0.08
    if corner == "lower left":
        x0 = 0.05
        x1 = min(x0 + frac, 0.98)
    else:
        x1 = 0.95
        x0 = max(x1 - frac, 0.02)
    ax.plot([x0, x1], [y, y], transform=ax.transAxes, color=color, lw=1.5,
            solid_capstyle="butt", clip_on=False, path_effects=[patheffects.withStroke(linewidth=2, foreground="black")])
    if length_um >= 1000:
        label = f"{length_um / 1000:g} mm"
    else:
        label = f"{length_um:g} " + r"$\mathrm{\mu}$m"
    ax.text((x0 + x1) / 2, y + 0.04, label, transform=ax.transAxes, color=color,
            fontsize=font_size, ha="center", va="bottom",
            path_effects=[patheffects.withStroke(linewidth=2, foreground="black")])
    ax.text((x0 + x1) / 2, y + 0.04, label, transform=ax.transAxes, color=color,
            fontsize=font_size, ha="center", va="bottom",
            path_effects=[patheffects.withStroke(linewidth=2, foreground="black")])


def draw_scale_bar(ax, image_width_px: int, model: HoloSystem, **kwargs) -> None:
    """Scale bar on a camera-pixel image."""
    draw_scale_bar_extent(
        ax, image_width_px * model.geometry.camera_pixel_pitch, **kwargs)


# ---------------------------------------------------------------------------
# Capture
# ---------------------------------------------------------------------------

@dataclass
class Capture:
    """
    One condition, scored on the camera grid.

    measured   - capture mean, own-max normalised, never resampled
    target     - target rendered to the camera grid through the registered
                  camera model
    mask       - field of view, minus the zero order, eroded
    rectified  - camera -> far field, display only
    """
    measured: np.ndarray
    target: np.ndarray
    mask: np.ndarray
    rectified: np.ndarray
    metrics: dict
    registration: Registration
    background: float
    clipped_fraction: float
    camera_shape: tuple

    def roi_metrics(self, roi) -> dict:
        validate_roi(roi, self.measured.shape, "ROI")
        return compute_metrics(crop_roi(self.measured, roi), crop_roi(self.target, roi),
                               crop_roi(self.mask, roi))

    def roi_is_valid(self, roi) -> bool:
        """False for an ROI that leaves the frame or the scored region."""
        if (roi.row0 < 0 or roi.col0 < 0
                or roi.row0 + roi.size > self.measured.shape[0]
                or roi.col0 + roi.size > self.measured.shape[1]):
            return False
        return bool(crop_roi(self.mask, roi).all())


def prepare_capture(base_dir, model: HoloSystem, target_path,
                    batch_size: int = 4, n_batches: int = None,
                    registration: Registration = None) -> Capture:
    """
    Capture directory -> scored Capture.

    `registration` should be the group-level offset: it is a property of the
    twin's gauge and the session, so fitting one per image fits noise and adds
    a free parameter per measurement. Passing None estimates it from this
    capture alone, which is useful only for measuring the group value.
    """
    raw = load_capture_mean(base_dir, batch_size, n_batches)
    camera_shape = tuple(raw.shape)
    clipped = float(np.mean(raw >= SATURATION_MARGIN * SATURATION_LEVEL))
    measured = raw / raw.max()

    target_far = load_target_far(model, target_path)
    reg = registration or estimate_target_shift(model, measured, target_far)

    with registered_camera(model, reg):
        target_cam = render_to_camera(model, target_far, camera_shape)
        mask = metric_mask(model, camera_shape)

    return Capture(
        measured=measured,
        target=target_cam,
        mask=mask,
        rectified=rectify_to_far_field(raw, model, square=True),
        metrics=compute_metrics(measured, target_cam, mask),
        registration=reg,
        background=background_residual(measured, target_cam, mask),
        clipped_fraction=clipped,
        camera_shape=camera_shape,
    )


def print_capture(label: str, cap: Capture) -> None:
    m = cap.metrics
    flags = ""
    if not cap.registration.ok:
        flags += "  REG?"
    if cap.clipped_fraction > 1e-4:
        flags += f"  CLIPPED {100 * cap.clipped_fraction:.2f}%"
    print(f"  {label}: NMSE={m['NMSE']:.4f}  PSNR={m['PSNR']:.2f} dB  SSIM={m['SSIM']:.4f}"
          f"  r={m['r']:.4f}  valid={100 * m['valid']:.1f}%"
          f"  {cap.registration}  bg={cap.background:+.4f}{flags}")


def top_down_to_axes(top_frac: float, h_frac: float) -> tuple:
    """axis_positions_mm returns (distance-from-top, size); add_axes wants bottom-up."""
    return 1 - top_frac - h_frac, h_frac

In [ ]:
from src.cgh.pipeline import cgh_predict_images

DEFAULT_EXPOSURE_TIME = 0.4
PREDICT_WITH_BACKGROUND = False   # the fitted stray-light field is mostly noise

# Exposures for captures taken away from the default. A linear difference is
# absorbed by the fitted scale; saturation is not.
EXPOSURE_TIMES = {"usaf": 0.01}


# The exposure a condition was captured at.
def exposure_for(label):
    return EXPOSURE_TIMES.get(label, DEFAULT_EXPOSURE_TIME)


# Holograms for one condition, stacked over batches to match the capture average.
def load_holograms(base_dir, n_batches=None):
    if n_batches is None:
        return np.load(Path(base_dir, "holograms.npy"))
    return np.concatenate(
        [np.load(Path(base_dir, f"batch_{n}", "holograms.npy"))
         for n in range(n_batches)], axis=0)


# Predicted intensity and predicted camera image for a set of holograms.
def predict_camera(model, holograms, camera_shape, registration=None,
                   exposure_time=DEFAULT_EXPOSURE_TIME):
    background = model.background
    was_active = getattr(background, "active", True)
    if not PREDICT_WITH_BACKGROUND:
        background.disable()
    try:
        with registered_camera(model, registration):
            intensity, camera = cgh_predict_images(
                model, torch.from_numpy(holograms),
                camera_shape=tuple(camera_shape), exposure_time=exposure_time)
    finally:
        if was_active and not PREDICT_WITH_BACKGROUND:
            background.enable()
    return to_numpy(intensity), to_numpy(camera)


# Panel letter and a short name above a plot axes.
def axes_panel_label(ax, letter, title=None, font_size=None, y=1.02):
    font_size = FONT_SIZE if font_size is None else font_size
    ax.text(0.0, y, letter, transform=ax.transAxes, fontsize=font_size + 1,
            fontweight="bold", ha="left", va="bottom")
    if title:
        ax.text(0.0, y, "    " + title, transform=ax.transAxes,
                fontsize=font_size, ha="left", va="bottom")


## Fig. 1 — optical system and model pipeline

Panel (a) is drawn externally. Panel (b) is rendered here from a synthetic
ground-truth model, so the schematic can be built without a fitted checkpoint.

In [ ]:
"""
Synthetic ground-truth model plus one CGH hologram, used to render the
pipeline schematic in Fig. 1b at a legible array size.
"""
from src.twin.model import OpticsGeometry
from src.training.build_true import build_ground_truth
from src.cgh.optimize import generate_holograms
from src.cgh.targets import load_target_from_image
from src.loss.losses import nmse_loss

DEMO_TARGET_PATH = TARGETS_DIR / "target_small.svg"

demo_geometry = OpticsGeometry(slm_pixels=(40, 60), wavelength=20,
                               camera_shape=(60, 60), camera_affine_supersample=4)
demo_model = build_ground_truth(demo_geometry, background_reflectance=0.5,
                                num_pupil_tiles=10, slm_field_noise_mag=0,
                                slm_field_scale=2.0, seed=0)
demo_model.report()
plt.show()

demo_target = load_target_from_image(str(DEMO_TARGET_PATH),
                                     shape=demo_geometry.far_fov_samples)

# CGH runs against the far-field intensity, so the camera model is disabled
# for the optimisation and re-enabled for the forward pass that follows.
demo_model.camera.disable()
demo_g, _ = generate_holograms(demo_model, target=demo_target, loss_fn=nmse_loss,
                               num_epochs=300, batch_size=1, micro_batch_size=1,
                               mask_centre=False, dark_penalty=False)
demo_model.camera.enable()

plt.imshow(demo_g[0].numpy(force=True), cmap="gray")
plt.show()
plt.imshow(demo_model(demo_g).mean(dim=0).numpy(force=True), cmap="gray")
plt.show()

In [ ]:
"""
Fig. 1b - the HoloSystem forward pass as a computational graph, for
composition with the optical-system schematic in panel (a).

Every intermediate array is taken from the model's own forward pass:

    g --[LUT]--> phi --[* kappa]--> phi_conv --[exp(i.) . E_SLM]--> E_up
      --[FT (+envelope)]--> U_ideal --[Psi]--> U_pupil --[+B]--> U
      --[|U|^2]--> I --[T (affine)]--> C

g, phi, phi_conv are real; E_up, U_ideal, U_pupil, U are complex (drawn as
stacked amplitude/phase cards); I, C are real intensities.
"""
import numpy as np
import torch
import matplotlib.transforms as mtransforms
from matplotlib.patches import Rectangle, Polygon
from matplotlib.colors import Normalize

FAR_CROP = None   # (H, W) centre-crop for the far-field panels, or None
CAM_CROP = None   # (H, W) centre-crop for the camera panel, or None

NODE_ORDER = ["g", "phi", "phi_conv", "E_up", "U_ideal", "U_pupil", "U", "I", "C"]

NODE_NAMES = {
    "g": "Hologram", "phi": "Selected phase", "phi_conv": "Convolved phase",
    "E_up": "Modulated field", "U_ideal": "Far field", "U_pupil": "Aberrated field",
    "U": "Total field", "I": "Intensity", "C": "Camera image",
}

NODE_TITLES = {
    "g": r"$g$", "phi": r"$\phi_\text{ideal}$", "phi_conv": r"$\phi$",
    "E_up": r"$E$", "U_ideal": r"$U_\text{ideal}$", "U_pupil": r"$U_\text{aberr}$",
    "U": r"$U$", "I": r"$I$", "C": r"$C$",
}

NODE_KIND = {
    "g": "real", "phi": "real", "phi_conv": "real",
    "E_up": "complex", "U_ideal": "complex", "U_pupil": "complex", "U": "complex",
    "I": "real", "C": "real",
}

# Paper notation: SLM grid P x Q, sub-pixel grid PK x QK, far field M x N,
# camera R x S. U_ideal is the extended pre-crop array, hence M_ext x N_ext.
NODE_DIM_SYMBOLS = {
    "g": r"$P \times Q$",
    "phi": r"$P \times Q$",
    "phi_conv": r"$PK \times QK$",
    "E_up": r"$PK \times QK$" + "\n(complex)",
    "U_ideal": r"$M_{\rm ext} \times N_{\rm ext}$" + "\n(complex)",
    "U_pupil": r"$M \times N$" + "\n(complex)",
    "U": r"$M \times N$" + "\n(complex)",
    "I": r"$M \times N$",
    "C": r"$R \times S$",
}

EDGE_NAMES = ["LUT", "Crosstalk", "SLM field", "Fourier transform",
              "Aberrations", "Stray light", "Intensity", "Camera model"]

EDGE_LABELS = [
    r"$\phi_\text{LUT}(g)$", r"$*\,\kappa$", r"$e^{j \phi}\!\cdot\!E_\text{SLM}$",
    r"$F\{E\}\cdot H$", r"$*\,\Psi$", r"$+\,B$", r"$|U|^2$", r"$T(I)$",
]

# Free-parameter count per stage, in paper notation (Table 1), not counted
# from the live model.
EDGE_PARAM_LABELS = [
    r"$N_{\rm LUT}$", r"$(K K_p)^2\!+\!2$", r"$PQ$" + "\n(complex)", r"$1$",
    r"$6$", r"$MN$" + "\n(complex)", r"", r"$5$",
]

# Nodes drawn square regardless of sample-count aspect: the far-field (u, v)
# domain is physically square even when sampled with M != N.
SQUARE_NODES = {"U_ideal", "U_pupil", "U", "I", "C"}
FAR_FIELD_KEYS = {"U_ideal", "U_pupil", "U", "I"}

# Isometric-slab rendering: each node is a real 2-D shear, so it reads as a
# leaning card. Real nodes are one card, complex nodes two (phase behind,
# amplitude in front). One shared arrow runs the length of the row.
SKEW_DEG_Y = 20
SHEAR_SCALE_X = 0.6
BOX_H_MAX = 0.90
BOX_H_MIN = 0.55
GAP_ARROW = 0.1
STACK_SHIFT = 0.14

WIDE_ARROW_GAP = 0.03
WIDE_ARROW_LABEL_H = 0.1
WIDE_ARROW_HALF_H = 0.15
WIDE_ARROW_HEAD_HALF_H = 0.25
WIDE_ARROW_HEAD_LEN = 0.45
WIDE_ARROW_PARAM_GAP = -0.06

NODE_NAME_Y = 0.0
NODE_DIM_Y = 0.2

# Dashed connector from the shared label baseline down to each card, so it is
# clear which label belongs to which node. Per-node gap, since card heights vary.
DASH_GAP_LABEL = 0.05
DASH_GAP_IMAGE = (0.05, 0.05, -0.15, -0.12, 0.05, 0.05, 0.05, 0.05, 0.05)
DASH_COLOR = "0.55"
DASH_LINEWIDTH = 0.6

IMSHOW_INTERP = "nearest"


def run_pipeline(model: HoloSystem, g: torch.Tensor) -> dict:
    """Replay propagate()/forward() step by step, keeping every intermediate."""
    g = g.unsqueeze(0).to(model.geometry.device)

    with torch.no_grad():
        phi = model.lut(g)                             # (1, P, Q) unwrapped phase
        E_in = model.slm_field()                       # (P, Q) complex incident field
        phi_conv = model.pixel._crosstalk_phase(phi)   # (1, PK, QK) unwrapped phase
        E_up = model.pixel._get_E_upsampled(phi, E_in)  # (1, PK, QK) complex
        U_ideal = model.pixel(phi, E_in)               # FT + envelope
        U_pupil = model.pupil(U_ideal)                 # aberrated + cropped
        U = U_pupil + model.background()
        I = model.get_intensity(U)
        C = model.camera(I)

    return dict(g=g[0], phi=phi[0], phi_conv=phi_conv[0], E_up=E_up[0],
                U_ideal=U_ideal[0], U_pupil=U_pupil[0], U=U[0], I=I[0], C=C[0])


def _centre_crop_shape(x: torch.Tensor, shape) -> torch.Tensor:
    """Centre-crop the last two dims down to `shape` (display only)."""
    h, w = x.shape[-2:]
    th, tw = shape
    if (h, w) == (th, tw):
        return x
    top, left = max((h - th) // 2, 0), max((w - tw) // 2, 0)
    return x[..., top:top + th, left:left + tw]


def _to_rgba(arr, cmap, vmin=None, vmax=None):
    if vmin is None or vmax is None:
        vmin = float(np.min(arr)) if vmin is None else vmin
        vmax = float(np.max(arr)) if vmax is None else vmax
        if vmax - vmin < 1e-12:
            vmax = vmin + 1e-12
    return plt.get_cmap(cmap)(Normalize(vmin=vmin, vmax=vmax)(arr))


def _shear_scale_pt(pt):
    """Numpy replica of the render transform, for layout bookkeeping."""
    x, y = pt
    return (SHEAR_SCALE_X * x, np.tan(np.radians(SKEW_DEG_Y)) * x + y)


def _node_transform(ax, x0, y0):
    return (mtransforms.Affine2D()
            .skew_deg(0, SKEW_DEG_Y)
            .scale(SHEAR_SCALE_X, 1.0)
            .translate(x0, y0) + ax.transData)


def _slab_bbox(w, h):
    """(min_x, max_x, min_y, max_y) of a card's footprint after shearing."""
    pts = [_shear_scale_pt(p) for p in [(0, 0), (w, 0), (w, h), (0, h)]]
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    return min(xs), max(xs), min(ys), max(ys)


def draw_slab(ax, arr, x0, y0, w, h, cmap, vmin=None, vmax=None, zorder_base=5):
    """One sheared card: field data plus a thin outline."""
    trans = _node_transform(ax, x0, y0)
    ax.imshow(_to_rgba(arr, cmap, vmin, vmax), extent=(0, w, 0, h), origin="upper",
              interpolation=IMSHOW_INTERP, transform=trans, zorder=zorder_base)
    ax.add_patch(Rectangle((0, 0), w, h, fill=False, transform=trans,
                           edgecolor="black", lw=0.5, zorder=zorder_base + 2))


def _effective_rows(shape, square):
    """Resolution proxy for card sizing: rows, or the geometric mean if square."""
    rows, cols = shape
    return float(np.sqrt(rows * cols)) if square else float(rows)


def node_box_size(shape, rows_ref, square=False):
    """Pre-shear (w, h) in canvas units, scaled by resolution and floored."""
    rows, cols = shape
    h = np.clip(BOX_H_MAX * np.sqrt(_effective_rows(shape, square) / rows_ref),
                BOX_H_MIN, BOX_H_MAX)
    return float(h if square else h * (cols / rows)), float(h)


def node_display_array(key, fields, far_crop, cam_crop):
    """The array actually drawn for a node, after any display crop."""
    arr = fields[key]
    if key in FAR_FIELD_KEYS and far_crop is not None:
        arr = _centre_crop_shape(arr, far_crop)
    elif key == "C" and cam_crop is not None:
        arr = _centre_crop_shape(arr, cam_crop)
    return arr


def pipeline_layout(fields, far_crop=None, cam_crop=None) -> dict:
    """
    Positions for the row of sheared cards, worked out without a figure so the
    caller can size a panel to the result before drawing into it.
    """
    arrays = {k: node_display_array(k, fields, far_crop, cam_crop) for k in NODE_ORDER}
    shapes = {k: tuple(v.shape[-2:]) for k, v in arrays.items()}
    rows_ref = max(_effective_rows(shapes[k], k in SQUARE_NODES) for k in NODE_ORDER)
    sizes = {k: node_box_size(shapes[k], rows_ref, k in SQUARE_NODES) for k in NODE_ORDER}

    # Lay out x-positions from each node's sheared footprint; complex nodes are
    # widened on the right by the amplitude card's offset.
    x_cursor = 0.0
    layout = {}
    y_max_seen, y_min_seen = -1e9, 1e9
    for key in NODE_ORDER:
        w, h = sizes[key]
        bbox = _slab_bbox(w, h)
        eff_bbox = ((bbox[0], bbox[1] + STACK_SHIFT, bbox[2], bbox[3])
                    if NODE_KIND[key] == "complex" else bbox)

        x0 = x_cursor - eff_bbox[0]
        layout[key] = dict(x0=x0, w=w, h=h, eff_bbox=eff_bbox)
        x_cursor = x0 + eff_bbox[1] + GAP_ARROW

        y0 = -h / 2
        y_max_seen = max(y_max_seen, y0 + eff_bbox[3])
        y_min_seen = min(y_min_seen, y0 + eff_bbox[2])

    canvas_w = x_cursor - GAP_ARROW + 0.30
    y_dims_bottom = y_min_seen - 0.24

    # Below the dimension labels, top to bottom: gap, arrow body and head,
    # gap, parameter-count text.
    arrow_y = y_dims_bottom - WIDE_ARROW_GAP - WIDE_ARROW_HALF_H
    param_label_y = arrow_y - WIDE_ARROW_HEAD_HALF_H - WIDE_ARROW_PARAM_GAP
    canvas_y0 = param_label_y - WIDE_ARROW_LABEL_H
    canvas_y1 = y_max_seen
    canvas_h = canvas_y1 - canvas_y0

    return dict(arrays=arrays, layout=layout, canvas_w=canvas_w, canvas_h=canvas_h,
                canvas_y0=canvas_y0, canvas_y1=canvas_y1, y_max_seen=y_max_seen,
                y_dims_bottom=y_dims_bottom, arrow_y=arrow_y,
                param_label_y=param_label_y)


def draw_pipeline(ax, prep: dict) -> None:
    """Render a pipeline_layout into `ax`, which must already be sized to
    prep["canvas_h"] / prep["canvas_w"]."""
    arrays, layout = prep["arrays"], prep["layout"]
    canvas_w, canvas_y0, canvas_y1 = (prep["canvas_w"], prep["canvas_y0"],
                                      prep["canvas_y1"])
    y_max_seen, y_dims_bottom = prep["y_max_seen"], prep["y_dims_bottom"]
    arrow_y, param_label_y = prep["arrow_y"], prep["param_label_y"]

    ax.set_xlim(-0.1, canvas_w)
    ax.set_ylim(canvas_y0, canvas_y1)
    ax.set_aspect("equal")
    ax.axis("off")

    node_left_x, node_right_x = {}, {}
    for i, key in enumerate(NODE_ORDER):
        L = layout[key]
        arr = arrays[key].squeeze().detach().cpu().numpy()
        y0 = -L["h"] / 2

        if NODE_KIND[key] == "real":
            draw_slab(ax, arr, L["x0"], y0, L["w"], L["h"], CMAP_GRAY)
        else:
            draw_slab(ax, np.angle(arr), L["x0"], y0, L["w"], L["h"], CMAP_PHASE,
                      vmin=-np.pi, vmax=np.pi, zorder_base=5)
            draw_slab(ax, np.abs(arr), L["x0"] + STACK_SHIFT, y0, L["w"], L["h"],
                      CMAP_AMP, zorder_base=10)

        x_center = L["x0"] + (L["eff_bbox"][0] + L["eff_bbox"][1]) / 2
        node_left_x[key] = L["x0"] + L["eff_bbox"][0]
        node_right_x[key] = L["x0"] + L["eff_bbox"][1]

        # Names and dimensions share one baseline across all nodes, so they
        # line up regardless of card height.
        ax.text(x_center, y_max_seen + NODE_NAME_Y, f"{NODE_NAMES[key]}\n{NODE_TITLES[key]}",
                ha="center", va="bottom", fontsize=FONT_SIZE, zorder=9,
                linespacing=1.35, multialignment="center")
        ax.text(x_center, y_dims_bottom + NODE_DIM_Y, NODE_DIM_SYMBOLS[key],
                ha="center", va="top", fontsize=FONT_SIZE, color="0.35", zorder=9)

        line_y_top = (y_max_seen + NODE_NAME_Y) - DASH_GAP_LABEL
        line_y_bottom = y0 + L["eff_bbox"][3] + DASH_GAP_IMAGE[i]
        if line_y_bottom < line_y_top:
            ax.plot([x_center, x_center], [line_y_bottom, line_y_top],
                    linestyle="--", color=DASH_COLOR, lw=DASH_LINEWIDTH, zorder=4)

    x_start, x_end = -0.09, canvas_w - 0.1
    head_len = min(WIDE_ARROW_HEAD_LEN, (x_end - x_start) * 0.05)
    ax.add_patch(Polygon([
        (x_start, arrow_y - WIDE_ARROW_HALF_H),
        (x_end - head_len, arrow_y - WIDE_ARROW_HALF_H),
        (x_end - head_len, arrow_y - WIDE_ARROW_HEAD_HALF_H),
        (x_end, arrow_y),
        (x_end - head_len, arrow_y + WIDE_ARROW_HEAD_HALF_H),
        (x_end - head_len, arrow_y + WIDE_ARROW_HALF_H),
        (x_start, arrow_y + WIDE_ARROW_HALF_H),
    ], closed=True, facecolor=ACCENT_COLOR, lw=0.8, zorder=3))

    for i in range(len(NODE_ORDER) - 1):
        xm = (node_right_x[NODE_ORDER[i]] + node_left_x[NODE_ORDER[i + 1]]) / 2
        ax.text(xm, arrow_y, f"{EDGE_NAMES[i]}\n{EDGE_LABELS[i]}",
                ha="center", va="center", fontsize=FONT_SIZE, zorder=8,
                linespacing=1.35, multialignment="center")
        if EDGE_PARAM_LABELS[i]:
            ax.text(xm, param_label_y, EDGE_PARAM_LABELS[i],
                    ha="center", va="top", fontsize=FONT_SIZE, color="0.4", zorder=8)

# ---------------------------------------------------------------------------
# Fig. 1 - (a) the optical schematic and (c) convergence share the top row;
# (b), the forward pass as a computational graph, spans the full width below.
#
# (a) is reserved rather than drawn: the schematic is composed externally and
# pasted into the gap, so only its letter is placed here.
#
# (c) also carries the sample sweep. Only each run's final validation loss is
# shown, as a column of points past the end of the curve: the curves themselves
# are near-identical in shape, so overlaying them adds ink without adding
# information, while where they end is the whole result.
# ---------------------------------------------------------------------------

FIG1_LOSS_HISTORY = FIT_DIR / "study_04_4pi_ap_fit/ap_4pi/optics_stage/loss_history.npy"
FIG1_SWEEP_DIR = PAPER_ROOT / "2026-08-13_twin_fit/study_07_sample_sweep"

# (label, history path, epoch to read). Two runs diverged after the epoch given
# and are read there rather than at the end.
FIG1_SWEEP_RUNS = [
    ("4",   FIG1_SWEEP_DIR / "4_samples/optics_stage/loss_history.npy",   200),
    ("8",   FIG1_SWEEP_DIR / "8_samples/optics_stage/loss_history.npy",   None),
    ("10",  FIG1_SWEEP_DIR / "10_samples/optics_stage/loss_history.npy",  190),
    ("20",  FIG1_SWEEP_DIR / "20_samples/optics_stage/loss_history.npy",  None),
    ("50",  FIG1_SWEEP_DIR / "50_samples/optics_stage/loss_history.npy",  None),
    ("100", FIG1_SWEEP_DIR / "100_samples/optics_stage/loss_history.npy", None),
    ("150", FIG1_LOSS_HISTORY,                                            None),
]

# Trained under different conditions, so it is not another point on the sample
# trend: it gets the accent colour rather than a place on the blue ramp.
FIG1_ASIDE_RUN = (r"20 (coarse $E_\text{SLM}$)",
                  FIG1_SWEEP_DIR / "20_samples_field_div40/optics_stage/loss_history.npy",
                  None)

FIG1_TOP_ROW_MM = 40.0       # height of the schematic / convergence row
FIG1_A_WIDTH_MM = 100.0      # reserved for the optical schematic
FIG1_ROW_GAP_MM = 4.0
FIG1_COL_GAP_MM = 4.0
FIG1_LETTER_MM = 4.0

FIG1_CONV_LEFT_MM = 11.0
FIG1_CONV_BOTTOM_MM = 9.0
FIG1_CONV_TOP_MM = 2.0
FIG1_CONV_RIGHT_MM = 2.0

FIG1_SWEEP_GAP_FRAC = 0.05   # white space either side of the divider, in epochs
FIG1_LEGEND_SPLIT = 0.8     # where the sweep labels start, in axes fraction


CONV_TRAIN_COLOR = "0.15"
CONV_VAL_COLOR = SECONDARY_COLOR


# Per-epoch history dict saved by train_model(), as float arrays.
def load_loss_history(path) -> dict:
    raw = np.load(str(path), allow_pickle=True)
    history = raw.item() if raw.ndim == 0 else dict(raw)
    return {k: np.asarray(v, dtype=np.float64) for k, v in history.items()}


# Final validation loss for one run, read at `epoch` or at the end.
def final_validation(path, epoch=None) -> float:
    history = load_loss_history(path)
    values = history["validation: structure (mean)"]
    return float(values[len(values) - 1 if epoch is None else min(epoch, len(values) - 1)])


# Blue accent at the smallest sample count, near-black at the largest.
def sweep_color(index, total):
    blue = np.array(mpl.colors.to_rgb(SECONDARY_COLOR))
    return tuple(blue * (1 - 0.9 * index / max(total - 1, 1)))


demo_model.eval()
fig1_fields = run_pipeline(demo_model, demo_g.squeeze(0))
fig1_prep = pipeline_layout(
    fig1_fields,
    far_crop=tuple(FAR_CROP) if FAR_CROP else None,
    cam_crop=tuple(CAM_CROP) if CAM_CROP else None,
)

fig1_history = load_loss_history(FIG1_LOSS_HISTORY)
conv_train = fig1_history["eval: structure"]
conv_epochs = np.arange(len(conv_train))
conv_val_min = fig1_history.get("validation: structure (min)")
conv_val_max = fig1_history.get("validation: structure (max)")
conv_val_mean = fig1_history.get("validation: structure (mean)")
conv_val_sem = fig1_history.get("validation: structure (sem)")

sweep_final = [(label, final_validation(path, epoch))
               for label, path, epoch in FIG1_SWEEP_RUNS]
aside_label, aside_path, aside_epoch = FIG1_ASIDE_RUN
aside_final = final_validation(aside_path, aside_epoch)


print(f"Loss history: {len(conv_epochs)} epochs")
if conv_val_mean is not None:
    gap = abs(conv_val_mean[-1] - conv_train[-1])
    print(f"  final train {conv_train[-1]:.5g}, validation {conv_val_mean[-1]:.5g} "
          f"+/- {conv_val_sem[-1]:.3g} (SEM), gap {gap / conv_val_sem[-1]:.2f} SEM")
print("Sample sweep, final validation loss:")
for label, value in sweep_final:
    print(f"  {label:>4s} samples: {value:.4f}")
print(f"  {aside_label:>12s}: {aside_final:.4f}")

fig1_pipe_mm = FIG_WIDTH_MM * fig1_prep["canvas_h"] / fig1_prep["canvas_w"]
fig1_height_mm = (FIG1_LETTER_MM + FIG1_TOP_ROW_MM + FIG1_ROW_GAP_MM
                  + FIG1_LETTER_MM + fig1_pipe_mm)

fig1 = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, fig1_height_mm / 25.4))


# Panel letter on its own baseline, so letters in a row line up.
def fig1_letter(x_mm, y_top_mm, text):
    ax = fig1.add_axes([x_mm / FIG_WIDTH_MM,
                        1 - (y_top_mm + FIG1_LETTER_MM) / fig1_height_mm,
                        0.1, FIG1_LETTER_MM / fig1_height_mm])
    ax.axis("off")
    ax.text(0, 0, text, transform=ax.transAxes, fontsize=FONT_SIZE + 1,
            fontweight="bold", ha="left", va="bottom")


fig1_letter(0, 0, "a")
fig1_letter(FIG1_A_WIDTH_MM + FIG1_COL_GAP_MM, 0, "c")

_conv_x0 = FIG1_A_WIDTH_MM + FIG1_COL_GAP_MM + FIG1_CONV_LEFT_MM
_conv_x1 = FIG_WIDTH_MM - FIG1_CONV_RIGHT_MM
_conv_top = FIG1_LETTER_MM + FIG1_CONV_TOP_MM
_conv_h = FIG1_TOP_ROW_MM - FIG1_CONV_TOP_MM - FIG1_CONV_BOTTOM_MM
ax_conv = fig1.add_axes([
    _conv_x0 / FIG_WIDTH_MM,
    1 - (_conv_top + _conv_h) / fig1_height_mm,
    (_conv_x1 - _conv_x0) / FIG_WIDTH_MM,
    _conv_h / fig1_height_mm,
])

_pipe_top = FIG1_LETTER_MM + FIG1_TOP_ROW_MM + FIG1_ROW_GAP_MM + FIG1_LETTER_MM
fig1_letter(0, FIG1_LETTER_MM + FIG1_TOP_ROW_MM + FIG1_ROW_GAP_MM, "b")
ax_pipe = fig1.add_axes([0, 1 - (_pipe_top + fig1_pipe_mm) / fig1_height_mm,
                         1, fig1_pipe_mm / fig1_height_mm])
draw_pipeline(ax_pipe, fig1_prep)

# The validation spread is drawn as a solid band rather than a translucent one:
# its mean tracks the training curve so closely that overlaying both only
# obscures the band. The regularised objective is left out for the same reason.
if conv_val_min is not None:
    ax_conv.fill_between(conv_epochs, conv_val_min, conv_val_max,
                         color=CONV_VAL_COLOR, lw=0, zorder=1,
                         label="Validation (min-max)")
ax_conv.plot(conv_epochs, conv_train, color=CONV_TRAIN_COLOR, lw=1.0, zorder=2,
             label="Train")

# Each sweep run's endpoint, stacked at one position past the curve so the
# column reads as a set of outcomes rather than as a continuation of it.
# The sweep points sit past a rule, with equal white space either side of it:
# they are outcomes of separate runs, not a continuation of the curve, and the
# gap is what says so.
last_epoch = conv_epochs[-1]
gap = last_epoch * FIG1_SWEEP_GAP_FRAC
divider_x = last_epoch + gap
sweep_x = last_epoch + 2 * gap
x_max = last_epoch + 3 * gap

sweep_handles = []
for index, (label, value) in enumerate(sweep_final):
    sweep_handles.append(
        ax_conv.plot([sweep_x], [value], "o", ms=3.5,
                     color=sweep_color(index, len(sweep_final)),
                     mec="white", mew=0.4, zorder=4, label=label)[0])
sweep_handles.append(
    ax_conv.plot([sweep_x], [aside_final], "D", ms=3.5, color=ACCENT_COLOR,
                 mec="white", mew=0.4, zorder=5, label=aside_label)[0])

ax_conv.axvline(divider_x, color="black", lw=0.8, zorder=3)

ax_conv.set_yscale("log")
ax_conv.set_xlim(conv_epochs[0], x_max)
ax_conv.set_xlabel("Epoch", fontsize=FONT_SIZE)
ax_conv.set_ylabel("Loss", fontsize=FONT_SIZE)
ax_conv.tick_params(labelsize=FONT_SIZE - 1, length=2)
ax_conv.grid(True, which="major", color="0.92", lw=0.5)
ax_conv.set_axisbelow(True)

# Both legends hang from the divider, so they stay clear of the points to its
# right and of the curve descending beneath them.
divider_frac = (divider_x - conv_epochs[0]) / (x_max - conv_epochs[0])
curve_legend = ax_conv.legend(
    handles=[h for h in ax_conv.get_lines() if h.get_label() == "Train"]
            + ([ax_conv.collections[0]] if conv_val_min is not None else []),
    loc="upper right", bbox_to_anchor=(divider_frac, 1.0),
    frameon=False, fontsize=FONT_SIZE - 1, handlelength=1.4,
    borderaxespad=0.2, labelspacing=0.3)
ax_conv.add_artist(curve_legend)
ax_conv.legend(handles=sweep_handles, loc="upper right",
               bbox_to_anchor=(divider_frac, FIG1_LEGEND_SPLIT), ncol=4,
               title="Final validation loss, by training samples",
               title_fontsize=FONT_SIZE - 1, alignment="right",
               frameon=False, fontsize=FONT_SIZE - 1, handlelength=0.8,
               handletextpad=0.3, columnspacing=0.7, labelspacing=0.3,
               borderaxespad=0.2)

save_figure(fig1, "fig1_pipeline", pad_inches=0.02)

## Fig. 2 — simultaneously recovered parameters

In [ ]:
"""
Fig. 2 - parameters recovered simultaneously by one fit of the 4pi,
aperture-fitted system: LUT, crosstalk kernel, pixel envelope, Seidel
aberrations, camera affine, and the complex SLM illumination field.

The vertical stack is specified in mm, so each row keeps its physical size
however the rest of the figure changes. Convergence lives in Fig. 1c.
"""
import numpy as np
import torch
import matplotlib.transforms as mtransforms
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.axes_size import Fixed
from matplotlib.patches import Rectangle

FIG2_CHECKPOINT = CKPT_4PI_AP

NOMINAL_LUT_SPAN = 4 * np.pi   # nominal total phase span of this system

CBAR_THICKNESS_IN = 0.07   # fixed physical colorbar size, so panels stay matched
CBAR_PAD_IN = 0.05
ANNOT_BBOX = dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5)


fig2_model = (HoloSystem.load(str(FIG2_CHECKPOINT), device=DEVICE)
              if FIG2_CHECKPOINT is not None else demo_model)
fig2_model.eval()


def format_pi(radians: float) -> str:
    """Phase as a multiple of pi, which is how modulation depth is quoted."""
    return rf"{radians / np.pi:.2f}$\pi$ rad"


def panel_label(fig, ax, letter, y_fig, title=None):
    """Panel letter, and a one-word name for what the panel shows, at a shared
    absolute figure height so labels in one row line up regardless of panel
    height; x follows each panel's own left edge."""
    trans = mtransforms.blended_transform_factory(ax.transAxes, fig.transFigure)
    ax.text(0, y_fig, letter, transform=trans, fontsize=FONT_SIZE + 1,
            fontweight="bold", va="bottom", ha="left")
    if title:
        ax.text(0, y_fig, "    " + title, transform=trans, fontsize=FONT_SIZE,
                va="bottom", ha="left")


def add_colorbar(fig, ax, im, label=None):
    div = make_axes_locatable(ax)
    div.set_anchor("N")   # top-anchor the image; append_axes ignores ax.set_anchor
    cax = div.append_axes("bottom", size=Fixed(CBAR_THICKNESS_IN), pad=Fixed(CBAR_PAD_IN))
    cbar = fig.colorbar(im, cax=cax, orientation="horizontal")
    cbar.ax.tick_params(labelsize=FONT_SIZE - 1)
    if label:
        cbar.set_label(label, fontsize=FONT_SIZE)
    return cbar




# ---------------------------------------------------------------------------
# Recovered parameters
# ---------------------------------------------------------------------------

with torch.no_grad():
    g_vals = torch.linspace(0, 1, 256, device=fig2_model.geometry.device)
    phi_lut = fig2_model.lut.phase(g_vals).cpu().numpy()
    g_np = g_vals.cpu().numpy()

    kernel = fig2_model.pixel.kernel.detach().cpu().numpy()
    grid_X = fig2_model.pixel._grid_X.detach().cpu().numpy()
    grid_Y = fig2_model.pixel._grid_Y.detach().cpu().numpy()

    # Compute FWHM directly from kernel data (1D slices through the peak)
    peak_y, peak_x = np.unravel_index(np.argmax(kernel), kernel.shape)
    
    x_coords = grid_X[peak_y, :] if grid_X.ndim == 2 else grid_X
    y_coords = grid_Y[:, peak_x] if grid_Y.ndim == 2 else grid_Y

    profile_x = kernel[peak_y, :]
    profile_y = kernel[:, peak_x]

    half_max_x = profile_x.max() / 2.0
    above_half_x = np.where(profile_x >= half_max_x)[0]
    fwhm_x = x_coords[above_half_x[-1]] - x_coords[above_half_x[0]]

    half_max_y = profile_y.max() / 2.0
    above_half_y = np.where(profile_y >= half_max_y)[0]
    fwhm_y = y_coords[above_half_y[-1]] - y_coords[above_half_y[0]]

    envelope_I = fig2_model.pixel.envelope.pow(2).detach().abs().cpu().numpy()
    fill = fig2_model.pixel.fill
    rho = fig2_model.pixel.deadspace_reflectance.abs().item()

    seidel = fig2_model.pupil.coeffs.detach().cpu().numpy()
    seidel_names = fig2_model.pupil.seidel_coeffs.NAMES

    E_slm = fig2_model.slm_field.field.detach().cpu().numpy()
    slm_amp, slm_phase = np.abs(E_slm), np.angle(E_slm)

    corners_cam, corners_fov = fig2_model.camera.get_corners()
    c_cam, c_fov = corners_cam.cpu().numpy(), corners_fov.cpu().numpy()
    affine_params = fig2_model.camera.affine_params

# ---------------------------------------------------------------------------
# Layout - vertical stack in mm, so rows 1 and 2 keep their physical sizes
# ---------------------------------------------------------------------------

FIG2_TOP_MM =  .0        # above row 1, for its panel letters
FIG2_ROW1_MM = 45.0       # a-e
FIG2_ROW2_MM = 64.0       # f, g
FIG2_BOTTOM_MM = 4.0      # below row 2's colorbar labels

FIG2_HEIGHT_MM = FIG2_TOP_MM + FIG2_ROW1_MM + FIG2_ROW2_MM + FIG2_BOTTOM_MM


def _from_top(mm: float) -> float:
    """Figure-fraction y of a point `mm` below the top edge."""
    return 1.0 - mm / FIG2_HEIGHT_MM


ROW1_TOP = _from_top(FIG2_TOP_MM)
ROW1_BOTTOM = _from_top(FIG2_TOP_MM + FIG2_ROW1_MM)
ROW2_TOP = ROW1_BOTTOM - 0.02
ROW2_BOTTOM = _from_top(FIG2_TOP_MM + FIG2_ROW1_MM + FIG2_ROW2_MM)

LABEL_Y_ROW1 = ROW1_TOP + 0.01
LABEL_Y_ROW2 = ROW2_TOP + 0.01

fig2 = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, FIG2_HEIGHT_MM / 25.4))

# Row 1 and row 2 get independent GridSpecs: row 2 has only two panels and
# needs a much smaller wspace than row 1's five to fill the same width. Within
# row 1 the gaps are not uniform either - the Seidel panel's rotated tick
# labels reach into its neighbours, so explicit spacer columns set each gap.
ROW1_GAP_NARROW = 0.6
ROW1_GAP_WIDE = 2.2
row1_width_ratios = [4, ROW1_GAP_NARROW, 4, ROW1_GAP_NARROW, 4,
                     ROW1_GAP_WIDE, 4, ROW1_GAP_WIDE, 4]
gs_top = fig2.add_gridspec(1, 9, width_ratios=row1_width_ratios, wspace=0,
                           left=0.06, right=0.985, top=ROW1_TOP, bottom=ROW1_BOTTOM)
gs_bot = fig2.add_gridspec(1, 2, wspace=0.08, left=0.005, right=0.999,
                           top=ROW2_TOP, bottom=ROW2_BOTTOM)

ax_lut = fig2.add_subplot(gs_top[0, 0])
ax_cross = fig2.add_subplot(gs_top[0, 2])
ax_env = fig2.add_subplot(gs_top[0, 4])
ax_seidel = fig2.add_subplot(gs_top[0, 6])
ax_affine = fig2.add_subplot(gs_top[0, 8])
ax_slm_amp = fig2.add_subplot(gs_bot[0, 0])
ax_slm_phase = fig2.add_subplot(gs_bot[0, 1])


# (a) LUT, recovered against the nominal linear ramp.
ax_lut.plot(g_np, phi_lut, color="black", lw=1.2, label="Recovered")
ax_lut.plot(g_np, g_np * NOMINAL_LUT_SPAN, "--", color="black", lw=1.0, label="Nominal")
ax_lut.set_xlabel("Grayscale", fontsize=FONT_SIZE)
ax_lut.set_ylabel("Phase (rad)", fontsize=FONT_SIZE)
ax_lut.tick_params(labelsize=FONT_SIZE)
ax_lut.legend(loc="lower right", frameon=False, fontsize=FONT_SIZE - 1,
              bbox_to_anchor=(1.05, 0))
ax_lut.text(0.05, 0.92, format_pi(phi_lut[-1] - phi_lut[0]), transform=ax_lut.transAxes,
            fontsize=FONT_SIZE, va="top", ha="left", bbox=ANNOT_BBOX)
ax_lut.set_box_aspect(1)
ax_lut.set_anchor("N")
panel_label(fig2, ax_lut, "a", LABEL_Y_ROW1, r"LUT ($\phi_\mathrm{LUT}$)")

# (b) Crosstalk kernel with the physical pixel boundary overlaid: there is no
# meaningful "nominal crosstalk" to compare against, since zero crosstalk is
# simply no kernel at all.
extent = (grid_X.min(), grid_X.max(), grid_Y.max(), grid_Y.min())
im_cross = ax_cross.imshow(kernel, cmap=CMAP_AMP, extent=extent, origin="upper")
for fn in (ax_cross.axvline, ax_cross.axhline):
    fn(-0.5, color=SECONDARY_COLOR, linestyle="--", lw=1)
    fn(0.5, color=SECONDARY_COLOR, linestyle="--", lw=1)
ax_cross.set_xticks([]); ax_cross.set_yticks([])
ax_cross.text(0.03, 0.03, rf"FWHM: $x$={fwhm_x:.2f}, $y$={fwhm_y:.2f} px",
              transform=ax_cross.transAxes, fontsize=FONT_SIZE - 1,
              ha="left", va="bottom", color="white")
add_colorbar(fig2, ax_cross, im_cross, "Weight")
panel_label(fig2, ax_cross, "b", LABEL_Y_ROW1, r"Crosstalk ($\kappa$)")

# (c) Pixel envelope intensity, with fill factor and dead-space reflectance.
im_env = ax_env.imshow(envelope_I, cmap=CMAP_GRAY, extent=(0, 1, 0, 1), aspect="equal")
ax_env.set_xticks([]); ax_env.set_yticks([])
# ax_env.text(0.03, 0.03, rf"$F$={fill**2:.2f}, $\rho$={rho:.2f}",
#             transform=ax_env.transAxes, fontsize=FONT_SIZE - 1,
#             ha="left", va="bottom", color="white")
add_colorbar(fig2, ax_env, im_env, "Normalised intensity")
panel_label(fig2, ax_env, "c", LABEL_Y_ROW1, "Envelope")

# (d) Seidel aberration coefficients.
x = np.arange(len(seidel))
ax_seidel.bar(x, seidel, color=SECONDARY_COLOR)
ax_seidel.set_xticks(x)
ax_seidel.set_xticklabels(seidel_names, rotation=45, ha="right",
                          rotation_mode="anchor", fontsize=FONT_SIZE - 1)
ax_seidel.set_ylabel("Value (rad)", fontsize=FONT_SIZE)
ax_seidel.tick_params(labelsize=FONT_SIZE)
ax_seidel.set_box_aspect(1)
ax_seidel.set_anchor("N")
panel_label(fig2, ax_seidel, "d", LABEL_Y_ROW1, r"Aberrations ($\Psi$)")

# (e) Camera field of view within the far-field field of view.
ax_affine.plot(c_fov[:, 0], c_fov[:, 1], "k-", lw=1, label="Far field")
ax_affine.plot(c_cam[:, 0], c_cam[:, 1], "r-", lw=1, label="Camera")
ax_affine.invert_yaxis()
ax_affine.set_xlabel(r"$u$ ($1/\Delta_\mathrm{SLM}$)", fontsize=FONT_SIZE)
ax_affine.set_ylabel(r"$v$ ($1/\Delta_\mathrm{SLM}$)", fontsize=FONT_SIZE, labelpad=0.5)
ax_affine.tick_params(labelsize=FONT_SIZE)
ax_affine.legend(fontsize=FONT_SIZE - 1, loc="upper center", frameon=False,
                 bbox_to_anchor=(0.5, 0.9))
ax_affine.set_box_aspect(1)
ax_affine.set_anchor("N")
p = affine_params
ax_affine.text(0.5, 0.15,
               rf"$\theta$={np.degrees(p['rotation']):.2f}°" + "\n" +
               rf"$s$=({p['scale'][0]:.3f}, {p['scale'][1]:.3f})" + "\n" +
               rf"$t$=({p['shift'][0]:.1f}, {p['shift'][1]:.1f}) px",
               transform=ax_affine.transAxes, fontsize=FONT_SIZE - 1,
               ha="center", va="bottom")
panel_label(fig2, ax_affine, "e", LABEL_Y_ROW1, r"Camera ($T$)")

# (f) SLM illumination field, phase kept wrapped. Amplitude and phase are two
# views of one recovered quantity, so they share a letter, and a region common
# to both is inset to show that its structure appears in each.
#
# The SLM's pixels are square, so this array's aspect is already correct and a
# square block of samples is a square region: the inset keeps that aspect
# rather than filling a fixed box, unlike the far-field insets elsewhere.
FIG2_FIELD_ROI_SPEC = (0.30, 0.77, 0.12)   # (row, col, size) as grid fractions
FIG2_INSET_FRAC = 0.4


def slm_field_region(shape: tuple, spec: tuple) -> tuple:
    fy, fx, size_frac = spec
    h, w = shape
    half = max(int(round(size_frac * min(h, w))) // 2, 1)
    cy, cx = int(round(fy * h)), int(round(fx * w))
    return (slice(cy - half, cy + half), slice(cx - half, cx + half))


def draw_square_inset(ax, image, cmap, color=SECONDARY_COLOR, **kwargs):
    """Inset that keeps its crop's aspect instead of filling a fixed box."""
    rows, cols = slm_field_region(image.shape, FIG2_FIELD_ROI_SPEC)
    inset = ax.inset_axes([1 - FIG2_INSET_FRAC, 0.0,
                           FIG2_INSET_FRAC, FIG2_INSET_FRAC])
    inset.imshow(image[rows, cols], cmap=cmap, interpolation="nearest", **kwargs)
    inset.set_xticks([]); inset.set_yticks([])
    inset.set_box_aspect(1)
    inset.set_anchor("SE")
    for spine in inset.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(1.2)
    return inset


def draw_region_box(ax, shape, color=SECONDARY_COLOR, linewidth=1.0):
    """Outline on the full image showing where the inset was taken from."""
    rows, cols = slm_field_region(shape, FIG2_FIELD_ROI_SPEC)
    ax.add_patch(Rectangle(
        (cols.start - 0.5, rows.start - 0.5),
        cols.stop - cols.start, rows.stop - rows.start,
        fill=False, edgecolor=color, lw=linewidth))


_rows, _cols = slm_field_region(slm_amp.shape, FIG2_FIELD_ROI_SPEC)

im_amp = ax_slm_amp.imshow(slm_amp, cmap=CMAP_AMP)
ax_slm_amp.set_xticks([]); ax_slm_amp.set_yticks([])
add_colorbar(fig2, ax_slm_amp, im_amp,
             r"Amplitude ($\sqrt{I_\mathrm{sat}}$)")
draw_region_box(ax_slm_amp, slm_amp.shape)
draw_square_inset(ax_slm_amp, slm_amp, CMAP_AMP)
panel_label(fig2, ax_slm_amp, "f", LABEL_Y_ROW2,
            r"SLM field ($|E_\mathrm{SLM}|$)")

im_phase = ax_slm_phase.imshow(slm_phase, cmap=CMAP_PHASE, vmin=-np.pi, vmax=np.pi)
ax_slm_phase.set_xticks([]); ax_slm_phase.set_yticks([])
add_colorbar(fig2, ax_slm_phase, im_phase, "Phase (rad)")
draw_region_box(ax_slm_phase, slm_phase.shape, color=ACCENT_COLOR)
draw_square_inset(ax_slm_phase, slm_phase, CMAP_PHASE, color=ACCENT_COLOR,
                  vmin=-np.pi, vmax=np.pi)
panel_label(fig2, ax_slm_phase, "g", LABEL_Y_ROW2,
            r"SLM field ($\angle E_\mathrm{SLM}$)")


save_figure(fig2, "fig2_recovered_params", pad_inches=0.1)

Supplementary figures - showing the other reports

In [ ]:
"""
Supplementary parameter figures.

The same layout as Fig. 2, for the two fits the main text does not show. Panel
letters are dropped - nothing in the text refers to a sub-panel - so each
panel is identified by the parameter it shows.

The second figure adds a row for the stray-light field: the recovered B in the
far field, then the same field taken back to the SLM plane. Both are cropped to
match Fig. 5, so the two figures can be read against each other; everything
outside those crops is noise.

Depends on the Fig. 2 cell for format_pi, add_colorbar, slm_field_region,
draw_square_inset, draw_region_box, ANNOT_BBOX and the row-1 gap constants.
"""
SI_TOP_MM = 12.0
SI_ROW1_MM = 45.0         # LUT, crosstalk, envelope, aberrations, camera
SI_ROW2_MM = 64.0         # SLM field amplitude and phase
SI_ROW3_MM = 46.0         # stray-light field, when shown
SI_ROW_GAP_MM = 2.0
SI_ROW3_GAP_MM = 10.0     # clears row 2's colorbar and its label
SI_ROW3_COL_GAP_MM = 2.0
SI_BOTTOM_MM = 4.0

# Same crops as Fig. 5: the far-field B as in (d, e), the SLM-plane view as in
# (b, c), so the supplementary figure lines up with it directly.
SI_B_CROP_SIZE = 500
SI_FT_CROP_W, SI_FT_CROP_H = 2340, 1740

# Kernel width at half maximum, along each axis through the peak. Reported
# instead of the fitted sigma once the kernel is no longer a bare Gaussian:
# the half-width is a property of the shape, whatever produced it.
def crosstalk_fwhm(kernel, grid_X, grid_Y) -> tuple:
    peak_y, peak_x = np.unravel_index(np.argmax(kernel), kernel.shape)
    x_coords = grid_X[peak_y, :] if grid_X.ndim == 2 else grid_X
    y_coords = grid_Y[:, peak_x] if grid_Y.ndim == 2 else grid_Y

    widths = []
    for profile, coords in ((kernel[peak_y, :], x_coords),
                            (kernel[:, peak_x], y_coords)):
        above = np.where(profile >= profile.max() / 2.0)[0]
        widths.append(float(coords[above[-1]] - coords[above[0]]))
    return widths[0], widths[1]


# An untrained field's phase spans far less than a cycle, so a cyclic map over
# the full turn shows almost nothing. A diverging map on the field's own
# symmetric range shows the structure that is there.
def phase_display(phase, wrapped: bool) -> dict:
    if wrapped:
        return dict(cmap=CMAP_PHASE, vmin=-np.pi, vmax=np.pi)
    return dict(cmap=CMAP_GRAY, vmin=np.min(phase), vmax=np.max(phase))

def recovered_parameters(model: HoloSystem) -> dict:
    """Everything the parameter figure draws, pulled off a fitted twin."""
    with torch.no_grad():
        g_vals = torch.linspace(0, 1, 256, device=model.geometry.device)
        corners_cam, corners_fov = model.camera.get_corners()
        return dict(
            g=g_vals.cpu().numpy(),
            lut=model.lut.phase(g_vals).cpu().numpy(),
            kernel=model.pixel.kernel.detach().cpu().numpy(),
            sigma=model.pixel.sigma.detach().cpu().numpy(),
            grid_X=model.pixel._grid_X.detach().cpu().numpy(),
            grid_Y=model.pixel._grid_Y.detach().cpu().numpy(),
            envelope=model.pixel.envelope.pow(2).detach().abs().cpu().numpy(),
            fill=model.pixel.fill,
            rho=model.pixel.deadspace_reflectance.abs().item(),
            seidel=model.pupil.coeffs.detach().cpu().numpy(),
            seidel_names=model.pupil.seidel_coeffs.NAMES,
            slm_field=model.slm_field.field.detach().cpu().numpy(),
            corners_cam=corners_cam.cpu().numpy(),
            corners_fov=corners_fov.cpu().numpy(),
            affine=model.camera.affine_params,
        )


def si_title(fig, ax, text, y_fig):
    """Parameter name above a panel, on a shared baseline across the row."""
    trans = mtransforms.blended_transform_factory(ax.transAxes, fig.transFigure)
    ax.text(0, y_fig, text, transform=trans, fontsize=FONT_SIZE,
            va="bottom", ha="left")


def si_field_pair(fig, axes, field, square=False, titles=("Amplitude", "Phase"),
                  y_label=None, wrapped_phase=True):
    """Amplitude and phase of one complex field, each with its own colorbar."""
    phase = np.angle(field)
    for ax, data, label, title, kwargs in (
            (axes[0], np.abs(field), "Amplitude (a.u.)", titles[0],
             dict(cmap=CMAP_AMP)),
            (axes[1], phase, "Phase (rad)", titles[1],
             phase_display(phase, wrapped_phase))):
        extent = dict(extent=(0, 1, 0, 1)) if square else {}
        im = ax.imshow(data, interpolation="nearest", **kwargs, **extent)
        ax.set_xticks([]); ax.set_yticks([])
        add_colorbar(fig, ax, im, label)
        if y_label is not None:
            si_title(fig, ax, title, y_label)


def build_parameter_figure(model: HoloSystem, name: str,
                           nominal_span: float = 4 * np.pi,
                           show_background: bool = False,
                           initial: bool = False):
    """
    Fig. 2's layout without the letters, optionally with a stray-light row.

    nominal_span is the system's design modulation depth, drawn as a straight
    ramp against the recovered LUT: the comparison is to the nominal linear
    response, not to a line through the recovered endpoints.

    Returns the figure; the caller saves it, so a variant can be inspected
    before it is written.
    """
    p = recovered_parameters(model)

    rows = [SI_ROW1_MM, SI_ROW2_MM] + ([SI_ROW3_MM] if show_background else [])
    gaps = [SI_ROW_GAP_MM, SI_ROW3_GAP_MM][:len(rows) - 1]
    height_mm = SI_TOP_MM + sum(rows) + sum(gaps) + SI_BOTTOM_MM

    def from_top(mm):
        return 1.0 - mm / height_mm

    tops, bottoms, cursor = [], [], SI_TOP_MM
    for k, h in enumerate(rows):
        tops.append(from_top(cursor))
        bottoms.append(from_top(cursor + h))
        cursor += h + (gaps[k] if k < len(gaps) else 0.0)

    fig = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, height_mm / 25.4))

    # Row 1's gaps are uneven: the Seidel panel's rotated tick labels reach into
    # its neighbours, so explicit spacer columns set each gap individually.
    gs_top = fig.add_gridspec(
        1, 9, width_ratios=[4, ROW1_GAP_NARROW, 4, ROW1_GAP_NARROW, 4,
                            ROW1_GAP_WIDE, 4, ROW1_GAP_WIDE, 4],
        wspace=0, left=0.06, right=0.985, top=tops[0], bottom=bottoms[0])
    gs_field = fig.add_gridspec(1, 2, wspace=0.08, left=0.005, right=0.999,
                                top=tops[1], bottom=bottoms[1])

    ax_lut, ax_cross, ax_env, ax_seidel, ax_affine = (
        fig.add_subplot(gs_top[0, c]) for c in (0, 2, 4, 6, 8))
    ax_amp, ax_phase = (fig.add_subplot(gs_field[0, c]) for c in (0, 1))

    y1, y2 = tops[0] + 0.008, tops[1] + 0.008

    # LUT, against the nominal linear ramp.
    ax_lut.plot(p["g"], p["lut"], color="black", lw=1.2, label="Recovered")
    ax_lut.plot(p["g"], p["g"] * nominal_span, "--", color="black", lw=1.0,
                label="Nominal")
    ax_lut.set_xlabel("Grayscale", fontsize=FONT_SIZE)
    ax_lut.set_ylabel("Phase (rad)", fontsize=FONT_SIZE)
    ax_lut.tick_params(labelsize=FONT_SIZE)
    if not initial:
        ax_lut.legend(loc="lower right", frameon=False, fontsize=FONT_SIZE - 1,
                      bbox_to_anchor=(1.05, 0))
    ax_lut.text(0.05, 0.92, format_pi(p["lut"][-1] - p["lut"][0]),
                transform=ax_lut.transAxes, fontsize=FONT_SIZE, va="top",
                ha="left", bbox=ANNOT_BBOX)
    ax_lut.set_box_aspect(1); ax_lut.set_anchor("N")
    si_title(fig, ax_lut, r"LUT ($\phi_\mathrm{LUT}$)", y1)

    # Crosstalk kernel, with the physical pixel boundary overlaid.
    extent = (p["grid_X"].min(), p["grid_X"].max(),
              p["grid_Y"].max(), p["grid_Y"].min())
    im = ax_cross.imshow(p["kernel"], cmap=CMAP_AMP, extent=extent, origin="upper")
    for fn in (ax_cross.axvline, ax_cross.axhline):
        fn(-0.5, color=SECONDARY_COLOR, linestyle="--", lw=1)
        fn(0.5, color=SECONDARY_COLOR, linestyle="--", lw=1)
    ax_cross.set_xticks([]); ax_cross.set_yticks([])
    if initial:
        kernel_text = (rf"$\sigma_x$={p['sigma'][1]:.2f}, "
                       rf"$\sigma_y$={p['sigma'][0]:.2f} px")
    else:
        fwhm_x, fwhm_y = crosstalk_fwhm(p["kernel"], p["grid_X"], p["grid_Y"])
        kernel_text = rf"FWHM: $x$={fwhm_x:.2f}, $y$={fwhm_y:.2f} px"
    ax_cross.text(0.03, 0.03, kernel_text, transform=ax_cross.transAxes,
                  fontsize=FONT_SIZE - 1, ha="left", va="bottom", color="white")
    add_colorbar(fig, ax_cross, im, "Weight")
    si_title(fig, ax_cross, r"Crosstalk ($\kappa$)", y1)

    # Pixel envelope, with fill factor and dead-space reflectance.
    im = ax_env.imshow(p["envelope"], cmap=CMAP_GRAY, extent=(0, 1, 0, 1),
                       aspect="equal")
    ax_env.set_xticks([]); ax_env.set_yticks([])
    ax_env.text(0.03, 0.03, rf"$F$={p['fill'] ** 2:.2f}, $\rho$={p['rho']:.2f}",
                transform=ax_env.transAxes, fontsize=FONT_SIZE - 1,
                ha="left", va="bottom", color="white")
    add_colorbar(fig, ax_env, im, "Normalised intensity")
    si_title(fig, ax_env, "Envelope", y1)

    # Seidel coefficients.
    x = np.arange(len(p["seidel"]))
    ax_seidel.bar(x, p["seidel"], color=SECONDARY_COLOR)
    ax_seidel.set_xticks(x)
    ax_seidel.set_xticklabels(p["seidel_names"], rotation=45, ha="right",
                              rotation_mode="anchor", fontsize=FONT_SIZE - 1)
    ax_seidel.set_ylabel("Value (rad)", fontsize=FONT_SIZE)
    ax_seidel.tick_params(labelsize=FONT_SIZE)
    ax_seidel.set_box_aspect(1); ax_seidel.set_anchor("N")
    si_title(fig, ax_seidel, r"Aberrations ($\Psi$)", y1)

    # Camera field of view within the far-field field of view.
    c_fov, c_cam, aff = p["corners_fov"], p["corners_cam"], p["affine"]
    ax_affine.plot(c_fov[:, 0], c_fov[:, 1], "k-", lw=1, label="Far field")
    ax_affine.plot(c_cam[:, 0], c_cam[:, 1], "r-", lw=1, label="Camera")
    ax_affine.invert_yaxis()
    ax_affine.set_xlabel(r"$u$ ($1/\Delta_\mathrm{SLM}$)", fontsize=FONT_SIZE)
    ax_affine.set_ylabel(r"$v$ ($1/\Delta_\mathrm{SLM}$)", fontsize=FONT_SIZE,
                         labelpad=0.5)
    ax_affine.tick_params(labelsize=FONT_SIZE)
    ax_affine.legend(fontsize=FONT_SIZE - 1, loc="upper center", frameon=False,
                     bbox_to_anchor=(0.5, 0.9))
    ax_affine.set_box_aspect(1); ax_affine.set_anchor("N")
    ax_affine.text(0.5, 0.15,
                   rf"$\theta$={np.degrees(aff['rotation']):.2f}°" + "\n" +
                   rf"$s$=({aff['scale'][0]:.3f}, {aff['scale'][1]:.3f})" + "\n" +
                   rf"$t$=({aff['shift'][0]:.1f}, {aff['shift'][1]:.1f}) px",
                   transform=ax_affine.transAxes, fontsize=FONT_SIZE - 1,
                   ha="center", va="bottom")
    si_title(fig, ax_affine, r"Camera ($T$)", y1)

    # SLM illumination field, with a shared region inset into both views.
    slm_amp, slm_phase = np.abs(p["slm_field"]), np.angle(p["slm_field"])
    im_amp = ax_amp.imshow(slm_amp, cmap=CMAP_AMP)
    ax_amp.set_xticks([]); ax_amp.set_yticks([])
    add_colorbar(fig, ax_amp, im_amp, r"Amplitude ($\sqrt{I_\mathrm{sat}}$)")
    draw_region_box(ax_amp, slm_amp.shape)
    draw_square_inset(ax_amp, slm_amp, CMAP_AMP)
    si_title(fig, ax_amp, r"SLM field ($|E_\mathrm{SLM}|$)", y2)

    im_phase = ax_phase.imshow(slm_phase, **phase_display(slm_phase, not initial))
    ax_phase.set_xticks([]); ax_phase.set_yticks([])
    add_colorbar(fig, ax_phase, im_phase, "Phase (rad)")
    draw_region_box(ax_phase, slm_phase.shape, color=ACCENT_COLOR)
    if not initial:
        draw_square_inset(ax_phase, slm_phase, CMAP_PHASE, color=ACCENT_COLOR,
                          vmin=-np.pi, vmax=np.pi)
    else:
        draw_square_inset(ax_phase, slm_phase, CMAP_GRAY, color=ACCENT_COLOR,
                          vmin=np.min(slm_phase), vmax=np.max(slm_phase))
    si_title(fig, ax_phase, r"SLM field ($\angle E_\mathrm{SLM}$)", y2)

    # Stray-light field: the far-field B, then the same field at the SLM plane,
    # where the aperture edge and the dust producing it are visible.
    #
    # Every panel is given the same image height and a width following its own
    # aspect, so the square far-field views and the wide SLM-plane ones read as
    # one row rather than as two different sizes.
    if show_background:
        e_slm, u_far = model.background.get_filtered(mode="both")
        u_far, e_slm = to_numpy(u_far), to_numpy(e_slm)
        u_far = center_crop(u_far, SI_B_CROP_SIZE,
                            int(SI_B_CROP_SIZE * u_far.shape[0] / u_far.shape[1]))
        e_slm = center_crop(e_slm, SI_FT_CROP_W, SI_FT_CROP_H)

        ft_aspect = SI_FT_CROP_W / SI_FT_CROP_H
        img_w = (FIG_WIDTH_MM - 3 * SI_ROW3_COL_GAP_MM) / (2 + 2 * ft_aspect)
        cols = axis_positions_mm(
            [img_w, img_w, img_w * ft_aspect, img_w * ft_aspect],
            SI_ROW3_COL_GAP_MM, FIG_WIDTH_MM)

        axes_back = []
        for x0, w in cols:
            ax = fig.add_axes([x0, bottoms[2], w, tops[2] - bottoms[2]])
            ax.set_anchor("N")
            axes_back.append(ax)

        y3 = tops[2] + 0.008
        si_field_pair(fig, axes_back[0:2], u_far, square=True, y_label=y3,
                      titles=(r"Stray light ($|B|$)", r"Stray light ($\angle B$)"))
        si_field_pair(fig, axes_back[2:4], e_slm, y_label=y3,
                      titles=(r"At the SLM ($|F\{B\}|$)",
                              r"At the SLM ($\angle F\{B\}$)"))

    save_figure(fig, name, pad_inches=0.1)
    return fig

si_model_4pi_init = HoloSystem.load(str(CKPT_4PI_AP_INIT), device=DEVICE)
si_model_4pi_init.eval()
build_parameter_figure(si_model_4pi_init, "figS_init_4pi",
                       nominal_span=4 * np.pi, show_background=True,
                       initial=True)


si_model_4pi = HoloSystem.load(str(CKPT_4PI_NO_AP), device=DEVICE)
si_model_4pi.eval()
build_parameter_figure(si_model_4pi, "figS_recovered_params_4pi",
                       nominal_span=4 * np.pi)

si_model_2pi = HoloSystem.load(str(CKPT_2PI_AP), device=DEVICE)
si_model_2pi.eval()
build_parameter_figure(si_model_2pi, "figS_recovered_params_2pi",
                       nominal_span=2 * np.pi, show_background=True)

Supplementary figure - showing the OTFs and PSFs of the pupil aberrations

In [ ]:
"""
Field-dependent aberration: the pupil's OTF per isoplanatic tile, and the point
spread function each one implies.

(a) the OTF phase, one tile per far-field node.
(b) the corresponding PSF amplitude, gamma-corrected so the shape rather than
    the peak sets what is visible.

Together these are the non-isoplanatic claim in one figure: if the aberration
were shift-invariant every tile would be identical, and a single kernel would
deconvolve the whole field. They are not, so it will not.

The OTF tiles keep their sampled aspect - they live on the SLM's rectangular
pupil - while the PSF tiles are drawn into a unit extent, since the far field
is square and only its sampling is rectangular.

The node grid is subsampled for display: at one panel per node the tiles are a
few pixels across and nothing is legible. Each PSF tile is cropped to its
centre, where all of its energy is.
"""
OTF_NODE_STRIDE = None      # None picks a stride giving about OTF_TARGET_NODES
OTF_TARGET_NODES = 7
PSF_CROP_FRAC = 0.35        # central fraction of each tile kept for the PSF
PSF_GAMMA = 0.7             # amplitude^gamma; below 1 lifts the wings
PSF_PER_TILE_NORM = False   # True compares shapes, False compares brightness

OTF_COL_GAP_MM = 2.0
OTF_LETTER_MM = 0.0         # letter band above each mosaic
OTF_XLABEL_MM = 8.0         # tick labels and the x-axis label
OTF_CBAR_GAP_MM = 5.0       # between the x-axis label and the colorbar
OTF_CBAR_MM = 7.0           # colorbar bar and its own label
OTF_LEFT_MM = 12.0          # y tick labels and the y-axis label, per panel


def tile_mosaic(field: torch.Tensor) -> np.ndarray:
    """
    (nodes, nodes, Ty, Tx) -> one image with the tiles laid out as a grid.

    A permute-reshape rather than an index loop: the axis order states which
    dimension becomes rows and which becomes columns, so a transposed mosaic
    fails visibly instead of silently.
    """
    n_y, n_x, t_y, t_x = field.shape
    return to_numpy(field.permute(0, 2, 1, 3).reshape(n_y * t_y, n_x * t_x))


def subsample_nodes(field: torch.Tensor, stride: int) -> tuple:
    """
    Every stride-th node in both directions, always keeping the first, last and
    central node, with the indices that were kept.

    The edges bound the field and the centre is where the aberration is
    weakest, so it is the reference the corners are read against; a stride that
    happens to skip it removes the comparison the figure exists to make.
    """
    def indices(n: int) -> list:
        return sorted(set(range(0, n, stride)) | {0, n - 1, (n - 1) // 2})

    idx_y, idx_x = indices(field.shape[0]), indices(field.shape[1])
    return field[idx_y][:, idx_x], idx_y, idx_x


def centre_crop_tiles(field: torch.Tensor, frac: float) -> torch.Tensor:
    """Keep the middle `frac` of each tile, where the PSF's energy sits."""
    t_y, t_x = field.shape[-2:]
    h, w = max(int(round(frac * t_y)) // 2, 1), max(int(round(frac * t_x)) // 2, 1)
    cy, cx = t_y // 2, t_x // 2
    return field[..., cy - h:cy + h, cx - w:cx + w]


otf = fig2_model.pupil.otf.detach()

# The shifts must name their axes: fftshift with no dim argument rolls the two
# node dimensions as well, which relabels every tile.
psf = torch.fft.fftshift(
    torch.fft.fft2(torch.fft.ifftshift(otf, dim=(-2, -1)),
                   dim=(-2, -1), norm="ortho"),
    dim=(-2, -1))

stride = OTF_NODE_STRIDE or max(1, round(otf.shape[0] / OTF_TARGET_NODES))
otf_shown, node_y, node_x = subsample_nodes(otf, stride)
psf_shown = centre_crop_tiles(subsample_nodes(psf, stride)[0], PSF_CROP_FRAC)

# Node positions across the field, spanning -0.5 to 0.5 in both axes with the
# centre at zero, so a tick reads as a place in the field rather than as an
# index into the tile grid.
node_v = np.linspace(-0.5, 0.5, otf.shape[0])[node_y]
node_u = np.linspace(-0.5, 0.5, otf.shape[1])[node_x]

print(f"OTF {tuple(otf.shape)} -> {len(node_y)}x{len(node_x)} nodes (stride {stride})")
print(f"  u {node_u[0]:.2f} to {node_u[-1]:.2f}, v {node_v[0]:.2f} to {node_v[-1]:.2f}")
print(f"  OTF tiles {tuple(otf_shown.shape[-2:])}, "
      f"PSF tiles {tuple(psf_shown.shape[-2:])}")

otf_phase = tile_mosaic(otf_shown.angle())

psf_amp = psf_shown.abs()
if PSF_PER_TILE_NORM:
    psf_amp = psf_amp / psf_amp.amax(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
else:
    psf_amp = psf_amp / psf_amp.max().clamp_min(1e-12)
psf_amp = tile_mosaic(psf_amp.pow(PSF_GAMMA))

# Side by side at a common image height, so the two mosaics sit level in the
# row and a tile in one lines up with the same tile in the other. Width then
# follows each panel's own aspect: the OTF tiles are rectangular, so that panel
# is the wider of the two.
otf_aspect = otf_phase.shape[1] / otf_phase.shape[0]   # width / height
psf_aspect = 1.0                                       # drawn into a unit extent

img_mm = ((FIG_WIDTH_MM - OTF_COL_GAP_MM - 2 * OTF_LEFT_MM)
          / (otf_aspect + psf_aspect))
otf_w_mm = (img_mm * otf_aspect, img_mm * psf_aspect)
otf_x0 = (OTF_LEFT_MM, 2 * OTF_LEFT_MM + otf_w_mm[0] + OTF_COL_GAP_MM)

height_mm = OTF_LETTER_MM + img_mm + OTF_XLABEL_MM + OTF_CBAR_GAP_MM + OTF_CBAR_MM

fig_otf = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, height_mm / 25.4))


def otf_panel(col, image, letter, title, cmap, label, square=False, **kwargs):
    """
    One mosaic in half the width, with its letter above and a colorbar below.

    square draws the tiles into a unit extent, for a field that is physically
    square but sampled on a rectangular grid; ticks then sit at tile centres in
    those units rather than in pixels.
    """
    x0 = otf_x0[col] / FIG_WIDTH_MM
    w = otf_w_mm[col] / FIG_WIDTH_MM
    n_y, n_x = len(node_y), len(node_x)

    ax_letter = fig_otf.add_axes([x0, 1 - OTF_LETTER_MM / height_mm, w,
                                  OTF_LETTER_MM / height_mm])
    ax_letter.axis("off")
    ax_letter.text(-0.1, 0, letter, transform=ax_letter.transAxes,
                   fontsize=FONT_SIZE + 1, fontweight="bold", ha="left", va="bottom")
    ax_letter.text(0, 0, "    " + title, transform=ax_letter.transAxes,
                   fontsize=FONT_SIZE, ha="left", va="bottom")

    ax = fig_otf.add_axes([x0, 1 - (OTF_LETTER_MM + img_mm) / height_mm, w,
                           img_mm / height_mm])
    ax.set_anchor("N")
    extent = dict(extent=(0, 1, 0, 1)) if square else {}
    im = ax.imshow(image, cmap=cmap, interpolation="nearest", **extent, **kwargs)

    if square:
        # Unit extent with origin upper: row 0 sits at y = 1.
        x_ticks = [(k + 0.5) / n_x for k in range(n_x)]
        y_ticks = [1 - (k + 0.5) / n_y for k in range(n_y)]
        v_lines = [k / n_x for k in range(1, n_x)]
        h_lines = [k / n_y for k in range(1, n_y)]
    else:
        t_y, t_x = image.shape[0] / n_y, image.shape[1] / n_x
        x_ticks = [(k + 0.5) * t_x - 0.5 for k in range(n_x)]
        y_ticks = [(k + 0.5) * t_y - 0.5 for k in range(n_y)]
        v_lines = [k * t_x - 0.5 for k in range(1, n_x)]
        h_lines = [k * t_y - 0.5 for k in range(1, n_y)]

    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f"{u:.2f}" for u in node_u], fontsize=FONT_SIZE - 1,
                       rotation=45, ha="right", rotation_mode="anchor")
    ax.set_xlabel(r"Field position $u$", fontsize=FONT_SIZE)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f"{v:.2f}" for v in node_v], fontsize=FONT_SIZE - 1)
    ax.set_ylabel(r"Field position $v$", fontsize=FONT_SIZE)
    ax.tick_params(labelsize=FONT_SIZE - 1, length=2)
    for side in ax.spines.values():
        side.set_linewidth(0.5)

    # Tile boundaries, so the mosaic reads as a grid of tiles rather than as one
    # textured image - the whole point being that the tiles differ.
    for x in v_lines:
        ax.axvline(x, color="white", lw=0.3, alpha=0.5)
    for y in h_lines:
        ax.axhline(y, color="white", lw=0.3, alpha=0.5)

    # Placed from the bottom of the figure, so it clears the x-axis label
    # instead of being positioned independently of it.
    cax = fig_otf.add_axes([x0 + 0.25 * w, OTF_CBAR_MM * 0.75 / height_mm,
                            0.5 * w, OTF_CBAR_MM * 0.15 / height_mm])
    cbar = fig_otf.colorbar(im, cax=cax, orientation="horizontal")
    cbar.set_label(label, fontsize=FONT_SIZE)
    cbar.ax.tick_params(labelsize=FONT_SIZE - 1, length=2)
    return ax


otf_panel(0, otf_phase, "a", r"OTF phase ($\angle\Psi$)",
          CMAP_PHASE, "Phase (rad)", vmin=-np.pi, vmax=np.pi)
otf_panel(1, psf_amp, "b", "PSF amplitude", CMAP_AMP,
          rf"Normalised amplitude ({PSF_GAMMA:g}$\gamma$)", square=True)

save_figure(fig_otf, "figS_otf_psf", pad_inches=0.05)

## Fig. 4 — target study

In [ ]:
"""
Fig. 4, data - six different target types, each an averaged camera capture
scored against its OWN target and its own radiometric gain.

Slow (model load, disk I/O, registration): run once. The figure cell below
reads only what is defined here.
"""
FIG4_BASE_DIR = CGH_DIR / "study_05_target_sweep"
FIG4_CHECKPOINT = CKPT_4PI_AP
FIG4_BATCH_SIZE = 4
FIG4_N_BATCHES = 5

FIG4_TARGETS = ["complex", "checkerboard_120", "checkerboard_10",
                "meta", "grayscale", "usaf"]
FIG4_LETTERS = "abcdef"

# Left None here: the checks cell measures one offset per acquisition group
# from these first-pass captures, then rescores them against it.
FIG_SESSION_SHIFT = None

fig4_model = HoloSystem.load(str(FIG4_CHECKPOINT), device=DEVICE)
assert_first_order(fig4_model, "Fig. 4:")

fig4_captures = {}
for name in FIG4_TARGETS:
    fig4_captures[name] = prepare_capture(
        FIG4_BASE_DIR / name, fig4_model, FIG4_BASE_DIR / name / "target.png",
        batch_size=FIG4_BATCH_SIZE, n_batches=FIG4_N_BATCHES,
        registration=FIG_SESSION_SHIFT,
    )

print(f"Prepared {len(fig4_captures)} target-type captures.")
for letter, name in zip(FIG4_LETTERS, FIG4_TARGETS):
    print_capture(f"{letter} ({name})", fig4_captures[name])

# A checkerboard finer than any panel shows, on the same region as (c). 
FIG4_FINE_NAME = "checkerboard_3"

fig4_fine_capture = prepare_capture(
    FIG4_BASE_DIR / FIG4_FINE_NAME, fig4_model,
    FIG4_BASE_DIR / FIG4_FINE_NAME / "target.png",
    batch_size=FIG4_BATCH_SIZE, n_batches=FIG4_N_BATCHES,
    registration=FIG_SESSION_SHIFT,
)
print_capture("fine (inset in c)", fig4_fine_capture)

In [ ]:
"""
Fig. 4, figure - one panel per target type, each with its own ROI inset.

Six different scenes, so each panel carries its own ROI position and size.
ROIs are placed as a fraction of the field of view and resolved per capture,
since captures of different sizes put the same physical region at different
pixel coordinates.

The inset is cropped from the raw capture: sensor data at its own resolution,
never interpolated, with a scale bar in micrometres at the sensor. The frame
behind it is the rectified view, where that region is a rotated quadrilateral,
drawn as an outline.

Fast: re-run this cell alone while dialling in the ROIs.
"""
FIG4_N_COLS = 3

# name: (row fraction, column fraction, size as a fraction of the FoV's short side)
FIG4_ROI_SPEC = {
    "complex":          (0.815, 0.86, 0.08),
    "checkerboard_120": (0.108, 0.108, 0.05),
    "checkerboard_10":  (0.13, 0.132, 0.02),
    "meta":             (0.122, 0.136, 0.032 ),
    "grayscale":        (0.37, 0.94, 0.1),
    "usaf":             (0.37, 0.632, 0.055),
}

FIG4_ROIS = {
    name: roi_from_fov(fig4_model, fig4_captures[name].camera_shape, *spec)
    for name, spec in FIG4_ROI_SPEC.items()
}

# The same field-of-view region as (c), resolved on this capture's own grid.
FIG4_FINE_PANEL = "c"
fig4_fine_roi = roi_from_fov(fig4_model, fig4_fine_capture.camera_shape,
                             *FIG4_ROI_SPEC["checkerboard_10"])
fig4_fine_inset = crop_roi(fig4_fine_capture.measured, fig4_fine_roi)

for name, roi in FIG4_ROIS.items():
    if not fig4_captures[name].roi_is_valid(roi):
        warnings.warn(f"Fig. 4 ROI for '{name}' reaches outside the scored region")


# The full field is the same in every panel, so one bar on (a) sets that scale
# for the figure. The insets differ in size from panel to panel, so each
# carries its own.
FIG4_FULL_BAR_PANEL = "a"


def fig4_scale_bar(ax, panel):
    draw_scale_bar(ax, panel.inset_image.shape[1], fig4_model)


FIG4_SHOW_TARGET_INSET = False


def fig4_annotate_image(ax, panel):
    """Scale bar on (a), the finer checkerboard on (c), and the target for the
    same region on every panel when that is switched on."""
    if panel.letter == FIG4_FULL_BAR_PANEL:
        draw_scale_bar_extent(ax, fov_extent_um(fig4_model, fig4_reference_shape))
    if panel.letter == FIG4_FINE_PANEL:
        draw_image_inset(ax, fig4_fine_inset, SECONDARY_COLOR, cmap=CMAP_GRAY,
                         width_frac=0.35, height_frac=0.35, margin_frac=0.05,
                         corner="lower right")
    if FIG4_SHOW_TARGET_INSET and panel.target_inset is not None:
        draw_image_inset(ax, panel.target_inset, SECONDARY_COLOR, cmap=CMAP_GRAY,
                         width_frac=0.35, height_frac=0.35, margin_frac=0.05,
                         corner="lower right")


fig4_reference_shape = fig4_captures[FIG4_TARGETS[0]].camera_shape


def fig4_panel(letter, name):
    cap = fig4_captures[name]
    roi = FIG4_ROIS[name]
    return Panel(
        image=cap.rectified,
        letter=letter,
        metrics_text=format_full_and_roi(cap.metrics, cap.roi_metrics(roi)),
        roi=roi,
        roi_color=ACCENT_COLOR,
        inset_image=crop_roi(cap.measured, roi),
        roi_outline=camera_roi_outline(roi, fig4_model, cap.camera_shape),
        annotate=fig4_scale_bar,
        annotate_image=fig4_annotate_image,
        target_inset=crop_roi(cap.target, roi),
    )


fig4_panels = [fig4_panel(letter, name)
               for letter, name in zip(FIG4_LETTERS, FIG4_TARGETS)]

build_grid_figure(
    fig4_panels, n_cols=FIG4_N_COLS, fig_width_mm=FIG_WIDTH_MM,
    out_path=fig_path("fig4_target_study"),
    col_gap_mm=1.0, row_gap_mm=0.0, sub_gap_mm=0.5,
    metrics_row_mm=6.0, letters_in_image=True,
    inset_width_frac=0.35, inset_height_frac=0.35, inset_margin_frac=0.05,
    inset_corner="lower left",
    font_size=FONT_SIZE, dpi=FIG_DPI,
)
print(f"Saved: {fig_path('fig4_target_study')}")
for name, roi in FIG4_ROIS.items():
    print(f"  {name:<18s} rows {roi.row0}-{roi.row0 + roi.size}, "
          f"cols {roi.col0}-{roi.col0 + roi.size}  "
          f"({roi_extent_um(roi, fig4_model):.0f} um across)")


## Fig. 4 — modulation depth and module ablation

In [ ]:
"""
Fig. 3, data - 2pi vs 4pi modulation, and the 4pi module ablation, all on one
shared scene and one shared target.

Two checkpoints are involved: conditions captured on the 2pi system are
aligned through the 2pi fit, everything else through the 4pi fit. Warping a
capture through the wrong camera model silently misplaces it, so the pairing
is set per condition rather than inferred from the path.

Slow (2 model loads, disk I/O, registration): run once.
"""
FIG3_TARGET_PATH = TARGETS_DIR / "complex.png"
FIG3_BATCH_SIZE = 4
FIG3_N_COLS = 5

# Left None here: the checks cell measures one offset per acquisition group
# from these first-pass captures, then rescores them against it.
FIG3_SESSION_SHIFT = None

# Row-major over a 2 x 5 grid; item i sits at (i // 5, i % 5).
#   "target"  - the shared target itself, not scored
#   "capture" - an averaged, aligned, registered, gain-matched capture
#   "empty"   - blank slot
#
# Row 1 is the argument for 4pi modulation: what the system does with no
# correction, what the best 2pi correction achieves, and what 4pi achieves,
# clamped and wrapped. Row 2 is the module ablation, all on 4pi + clamping.
FIG3_ITEMS = [
    {"kind": "target", "name": "target", "title": "Target"},
    {"kind": "capture", "name": "2pi, uncorrected", "checkpoint": "2pi",
     "title": "Baseline\nuncorrected, 2π",
     "dir": CROSSTALK_DIR / "ap_2pi/tmbgd_no_correction"},
    {"kind": "capture", "name": "2pi, corrected", "checkpoint": "2pi",
     "title": "Baseline\ncorrected, 2π",
     "dir": CROSSTALK_DIR / "ap_2pi/complex_corrected"},
    {"kind": "capture", "name": "4pi, clamped", "checkpoint": "4pi",
     "title": "Full model\nclamped, 4π",
     "dir": CGH_DIR / "study_05_target_sweep/complex"},
    {"kind": "capture", "name": "4pi, wrapped", "checkpoint": "4pi",
     "title": "Full model\nwrapped, 4π",
     "dir": CROSSTALK_DIR / "ap_4pi/tmbgd_full_corr_wrap"},
    {"kind": "capture", "name": "crosstalk off", "checkpoint": "4pi",
     "title": "Crosstalk off",
     "dir": CGH_DIR / "study_06_module_ablation/pixel_off/complex"},
    {"kind": "capture", "name": "SLM field off", "checkpoint": "4pi",
     "title": "SLM field off",
     "dir": CGH_DIR / "study_06_module_ablation/slm_field_off/complex"},
    {"kind": "capture", "name": "aberrations off", "checkpoint": "4pi",
     "title": "Aberrations off",
     "dir": CGH_DIR / "study_06_module_ablation/pupil_off/complex"},
    {"kind": "capture", "name": "LUT off", "checkpoint": "4pi",
     "title": "LUT off",
     "dir": CGH_DIR / "study_06_module_ablation/lut_off/complex"},
    {"kind": "empty", "name": None},
]

fig3_models = {
    "2pi": HoloSystem.load(str(CKPT_2PI_AP), device=DEVICE),
    "4pi": HoloSystem.load(str(CKPT_4PI_AP), device=DEVICE),
}
for tag, m in fig3_models.items():
    assert_first_order(m, f"Fig. 3 ({tag}):")

# The target on the far-field grid, so the reference panel shares a grid with
# the rectified captures beside it. Not scored.
fig3_target_image = load_target_image(FIG3_TARGET_PATH,
                                      fig3_models["4pi"].geometry.far_fov_samples)

fig3_captures = [None] * len(FIG3_ITEMS)
for i, item in enumerate(FIG3_ITEMS):
    if item["kind"] != "capture":
        continue
    fig3_captures[i] = prepare_capture(
        item["dir"], fig3_models[item["checkpoint"]], FIG3_TARGET_PATH,
        batch_size=FIG3_BATCH_SIZE, n_batches=1,
        registration=FIG3_SESSION_SHIFT,
    )

n_captures = sum(1 for it in FIG3_ITEMS if it["kind"] == "capture")
n_empty = sum(1 for it in FIG3_ITEMS if it["kind"] == "empty")
print(f"Prepared {n_captures} scored captures + 1 target ({n_empty} empty slot(s)).")
for i, item in enumerate(FIG3_ITEMS):
    r, c = divmod(i, FIG3_N_COLS)
    if item["kind"] == "empty":
        print(f"  [{r},{c}] - (empty)")
    elif item["kind"] == "target":
        print(f"  [{r},{c}] {item['name']}: {fig3_target_image.shape} (reference, not scored)")
    else:
        print_capture(f"[{r},{c}] {item['name']} ({item['checkpoint']})", fig3_captures[i])

In [ ]:
"""
Fig. 3, figure - the target and seven conditions, in two rows of four.

Each condition is the full rectified frame with the region of interest inset
from the raw capture, and the metrics for both underneath. The target in (a)
already shows what the region should look like, so the frames carry no
reference box. The
summary charts are gone: Fig. 7 carries the quantitative comparison, and
repeating it here only competed with the images.

ROIs are in camera pixels, resolved per capture from a field-of-view fraction:
the 2pi and 4pi conditions were recorded on different sensor windows, so one
pixel box cannot land on the same physical region in both. The target has no
capture grid of its own, so it is rendered through the reference condition's
camera and cropped with that condition's ROI.
"""
FIG3_ROI_SPEC = (0.285, 0.876, 0.125)
FIG3_N_COLS = 4
FIG3_FULL_MODE = "image"           # "image" | "residual"
FIG3_RESIDUAL_PERCENTILE = 99.0

FIG3_COL_GAP_MM = 1.0
FIG3_ROW_GAP_MM = 1.0
FIG3_SUB_GAP_MM = 0.5
FIG3_METRICS_MM = 6.0
FIG3_INSET_FRAC = 0.5
FIG3_INSET_MARGIN = 0.0

# Drop the wrapped-phase condition: its message is about CGH bookkeeping rather
# than about the model, and it costs a panel the ablation needs.
FIG3_SKIP = ("4pi, wrapped",)

fig3_panel_mm = (FIG_WIDTH_MM - (FIG3_N_COLS - 1) * FIG3_COL_GAP_MM) / FIG3_N_COLS
fig3_cols = axis_positions_mm([fig3_panel_mm] * FIG3_N_COLS,
                              FIG3_COL_GAP_MM, FIG_WIDTH_MM)

_sizes = [fig3_panel_mm, FIG3_METRICS_MM] * 2
_gaps = [FIG3_SUB_GAP_MM, FIG3_ROW_GAP_MM, FIG3_SUB_GAP_MM]
fig3_height_mm = sum(_sizes) + sum(_gaps)
fig3_v = axis_positions_mm(_sizes, _gaps, fig3_height_mm)

fig3_shown = [i for i, item in enumerate(FIG3_ITEMS)
              if item["kind"] == "capture" and fig3_captures[i] is not None
              and item["name"] not in FIG3_SKIP]
# The target leads, so the reader has the reference before the comparisons.
fig3_letters = {i: letter for i, letter in zip(fig3_shown, "bcdefgh")}


def fig3_roi(i):
    return roi_from_fov(fig3_models[FIG3_ITEMS[i]["checkpoint"]],
                        fig3_captures[i].camera_shape, *FIG3_ROI_SPEC)


fig3_ref_index = fig3_shown[0]
fig3_ref_model = fig3_models[FIG3_ITEMS[fig3_ref_index]["checkpoint"]]
fig3_ref_capture = fig3_captures[fig3_ref_index]
fig3_ref_roi = fig3_roi(fig3_ref_index)

with registered_camera(fig3_ref_model, fig3_ref_capture.registration):
    fig3_target_camera = render_to_camera(
        fig3_ref_model, fig3_target_image, fig3_ref_capture.camera_shape)


def fig3_residual(cap) -> np.ndarray:
    return rectify_to_far_field(cap.measured - cap.target, fig3_ref_model,
                                normalize=False, square=True)


fig3_residuals = {i: fig3_residual(fig3_captures[i]) for i in fig3_shown}
fig3_res_lim = float(np.percentile(
    np.abs(np.concatenate([r.ravel() for r in fig3_residuals.values()])),
    FIG3_RESIDUAL_PERCENTILE))

fig3 = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, fig3_height_mm / 25.4))


def fig3_axes(col, v_index):
    x0, w = fig3_cols[col]
    bottom, h = top_down_to_axes(*fig3_v[v_index])
    ax = fig3.add_axes([x0, bottom, w, h])
    ax.axis("off")
    return ax


def fig3_image_panel(col, v_index, image, inset, letter, title, model,
                     roi=None, scale_bars=False, residual=False, outline=None):
    ax = fig3_axes(col, v_index)
    if residual:
        ax.imshow(image, cmap=CMAP_RESIDUAL, interpolation="nearest",
                  vmin=-fig3_res_lim, vmax=fig3_res_lim)
    else:
        ax.imshow(image, cmap=CMAP_GRAY, interpolation="nearest")
    ax.set_box_aspect(1)
    if outline is not None:
        draw_roi_outline(ax, outline, ACCENT_COLOR)
    inset_ax = draw_image_inset(
        ax, inset, ACCENT_COLOR, cmap=CMAP_GRAY, width_frac=FIG3_INSET_FRAC,
        height_frac=FIG3_INSET_FRAC, margin_frac=FIG3_INSET_MARGIN,
        corner="lower left")
    draw_panel_letter(ax, letter, FONT_SIZE - 1, title)
    if scale_bars:
        draw_scale_bar_extent(ax, fov_extent_um(model, fig3_ref_capture.camera_shape))
        draw_scale_bar_extent(inset_ax, roi_extent_um(roi, model))
    return ax


# (a) the target, on the reference condition's camera grid so its inset is the
# same physical region as every other panel's.
fig3_image_panel(
    0, 0, square_for_display(fig3_target_image, fig3_ref_model),
    crop_roi(fig3_target_camera, fig3_ref_roi), "a", FIG3_ITEMS[0]["title"],
    fig3_ref_model, roi=fig3_ref_roi, scale_bars=True,
    outline=camera_roi_outline(fig3_ref_roi, fig3_ref_model,
                               fig3_ref_capture.camera_shape))

for k, i in enumerate(fig3_shown):
    row, col = divmod(k + 1, FIG3_N_COLS)
    v_img, v_met = 2 * row, 2 * row + 1
    cap, item, roi = fig3_captures[i], FIG3_ITEMS[i], fig3_roi(i)
    model = fig3_models[item["checkpoint"]]
    residual = FIG3_FULL_MODE == "residual"

    fig3_image_panel(
        col, v_img, fig3_residuals[i] if residual else cap.rectified,
        crop_roi(cap.measured, roi), fig3_letters[i], item["title"],
        model,
        roi=roi, residual=residual)

    ax_txt = fig3_axes(col, v_met)
    ax_txt.text(0.5, 1.0, format_full_and_roi(cap.metrics, cap.roi_metrics(roi)),
                transform=ax_txt.transAxes, fontsize=FONT_SIZE - 1,
                ha="center", va="top", linespacing=1.4)

save_figure(fig3, "fig3_ablation")
for i in fig3_shown:
    rm = fig3_captures[i].roi_metrics(fig3_roi(i))
    print(f"  ({fig3_letters[i]}) {FIG3_ITEMS[i]['name']}: "
          f"full NMSE={fig3_captures[i].metrics['NMSE']:.4f}  "
          f"ROI NMSE={rm['NMSE']:.4f}  PSNR={rm['PSNR']:.2f} dB")

## Checks

Run after Figs. 3 and 4, before Figs. 5 and 6.

In [ ]:
"""
Checks to run before trusting any number.

  1. Does folding a registration offset into the affine remove it or double
     it? A sign error in the fold looks like a doubled offset, not a crash.
  2. Do the per-condition offsets cluster within an acquisition group? If they
     do, the offset is the twin's tip/tilt-against-camera-translation gauge, a
     property of the session, and one value should serve the whole group.
  3. Did any capture clip? A linear exposure difference is absorbed by the
     fitted scale; clipped pixels are not recoverable by any scalar.
  4. How much of each frame is scored, after the field of view, the zero order
     and the erosion are accounted for?
"""
# Registration is shared over an ACQUISITION: captures taken in one session
# share the twin's gauge offset, while two sessions need not. The mapping is by
# the study directory a capture came from, since that is what a session is.
ACQUISITION_ROOTS = {
    "study_05_target_sweep": "sweep",
    "study_06_module_ablation": "ablation",
    "ap_2pi": "crosstalk-2pi",
    "ap_4pi": "crosstalk-4pi",
}


def acquisition_key(path) -> str:
    """Session label for a capture directory."""
    node = Path(path)
    for part in [node.name] + [p.name for p in node.parents]:
        if part in ACQUISITION_ROOTS:
            return ACQUISITION_ROOTS[part]
    return "other"


def verify_shift_sign(model: HoloSystem, cap: Capture, target_path) -> None:
    """
    Re-measure the offset after folding it in. It should collapse towards zero;
    if it roughly doubles, SHIFT_SIGN has the wrong sign.
    """
    target_far = load_target_far(model, target_path)
    before = estimate_target_shift(model, cap.measured, target_far)
    with registered_camera(model, before):
        rendered = render_to_camera(model, target_far, cap.camera_shape)
    after = estimate_shift(rectify_to_far_field(rendered, model),
                           rectify_to_far_field(cap.measured, model))

    print(f"shift fold: before {before}, after {after}")
    if after.magnitude() > before.magnitude():
        print(f"  -> residual grew: set SHIFT_SIGN = {-SHIFT_SIGN:+.0f} and re-run")
    else:
        print(f"  -> residual fell from {before.magnitude():.2f} to "
              f"{after.magnitude():.2f} px; sign is correct")


def group_shift(conditions: list, group: str, exclude_unreliable: bool = True):
    """
    One registration for an acquisition group, as the median of its members'
    offsets. Conditions whose own estimate was flagged are excluded from the
    median but still inherit the result.
    """
    members = [c for c in conditions if c["group_key"] == group]
    if not members:
        return None
    regs = [c["capture"].registration for c in members]
    usable = [r for r in regs if r.ok] if exclude_unreliable else regs
    if not usable:
        warnings.warn(f"group '{group}': no reliable estimate; using all members")
        usable = regs

    dys = np.array([r.dy for r in usable])
    dxs = np.array([r.dx for r in usable])
    spread = float(np.hypot(dys.std(ddof=1) if len(dys) > 1 else 0.0,
                            dxs.std(ddof=1) if len(dxs) > 1 else 0.0))
    print(f"group '{group}': n={len(usable)}/{len(regs)}  "
          f"dy {np.median(dys):+.2f}  dx {np.median(dxs):+.2f}  spread {spread:.2f} px"
          + ("  (tight - use one value for the group)" if spread < 0.5
             else "  (spread too large for one value)"))
    return Registration(dy=float(np.median(dys)), dx=float(np.median(dxs)),
                        ambiguity=0.0, ok=spread < 0.5)


def check_conditions(conditions: list) -> dict:
    """Per-condition report, then one registration per acquisition group."""
    header = (f"{'condition':<26s} {'group':>6s} {'clip%':>7s} {'dy':>7s} {'dx':>7s} "
              f"{'amb':>6s} {'valid%':>8s}  flags")
    print(header)
    print("-" * len(header))
    for cond in conditions:
        cap = cond["capture"]
        reg = cap.registration
        flags = []
        if cap.clipped_fraction > 1e-4:
            flags.append("CLIPPED")
        if not reg.ok:
            flags.append("REG-UNRELIABLE")
        print(f"{cond['label']:<26s} {cond['group_key']:>6s} "
              f"{100 * cap.clipped_fraction:7.3f} {reg.dy:+7.2f} {reg.dx:+7.2f} "
              f"{reg.ambiguity:6.2f} {100 * cap.metrics['valid']:8.1f}  {' '.join(flags)}")

    print()
    groups = sorted({c["group_key"] for c in conditions})
    return {g: group_shift(conditions, g) for g in groups}


def check_roi(conditions: list, roi) -> None:
    """Confirm an ROI lies inside every condition's scored region."""
    print(f"ROI rows {roi.row0}-{roi.row0 + roi.size}, "
          f"cols {roi.col0}-{roi.col0 + roi.size}")
    for cond in conditions:
        cap = cond["capture"]
        frac = float(crop_roi(cap.mask, roi).mean())
        print(f"  {cond['label']:<26s} "
              f"{'ok' if frac == 1.0 else f'PARTIAL ({100 * frac:.1f}% valid)'}")


In [ ]:
"""
The shared condition list: every scored capture in the paper, with what is
needed to re-score or predict it.

`group` labels a condition for presentation (which panel group it belongs to).
`group_key` labels the acquisition it came from, and is what registration is
shared over: the 2pi and 4pi captures were taken through different fits, so
each carries its own offset.

Append to the checks cell. Requires the Fig. 3 and Fig. 4 data cells to have run.
"""

def fig4_group_of(item: dict) -> str:
    """Presentation group for a Fig. 4 condition, from its panel title."""
    title = item.get("title", "")
    if "Baseline" in title:
        return "Baselines"
    if "Full model" in title:
        return "Full model"
    return "Ablations"


def build_conditions() -> list:
    conditions = []

    for name in FIG3_TARGETS:
        conditions.append(dict(
            label=name,
            group="Target types",
            group_key=acquisition_key(FIG3_BASE_DIR / name),
            capture=fig3_captures[name],
            model=fig3_model,
            dir=FIG3_BASE_DIR / name,
            target_path=FIG3_BASE_DIR / name / "target.png",
            n_batches=FIG3_N_BATCHES,
            batch_size=FIG3_BATCH_SIZE,
            exposure_time=exposure_for(name),
        ))

    for i, item in enumerate(FIG4_ITEMS):
        if item["kind"] != "capture" or fig4_captures[i] is None:
            continue
        conditions.append(dict(
            label=item["name"],
            group=fig4_group_of(item),
            group_key=acquisition_key(item["dir"]),
            capture=fig4_captures[i],
            model=fig4_models[item["checkpoint"]],
            dir=item["dir"],
            target_path=FIG4_TARGET_PATH,
            n_batches=1,
            batch_size=FIG4_BATCH_SIZE,
            exposure_time=exposure_for(item["name"]),
        ))

    return conditions


def rescore_conditions(conditions: list, group_shifts: dict) -> list:
    """
    Re-prepare every capture against its group's registration, replacing the
    per-condition estimates used to measure those group values.
    """
    for cond in conditions:
        reg = group_shifts.get(cond["group_key"])
        if reg is None:
            warnings.warn(f"no group shift for '{cond['group_key']}'; "
                          f"leaving '{cond['label']}' on its own estimate")
            continue
        cond["capture"] = prepare_capture(
            cond["dir"], cond["model"], cond["target_path"],
            batch_size=cond["batch_size"], n_batches=cond["n_batches"],
            registration=reg,
        )
    return conditions


def sync_figure_captures(conditions: list) -> None:
    """Push rescored captures back into the Fig. 3 and Fig. 4 containers."""
    by_label = {c["label"]: c["capture"] for c in conditions}
    for name in FIG3_TARGETS:
        if name in by_label:
            fig3_captures[name] = by_label[name]
    for i, item in enumerate(FIG4_ITEMS):
        if item["kind"] == "capture" and item["name"] in by_label:
            fig4_captures[i] = by_label[item["name"]]


# Pass 1: per-condition estimates, to measure the group offsets.
scored_conditions = build_conditions()
group_shifts = check_conditions(scored_conditions)

# Pass 2: rescore everything against one offset per acquisition group.
print()
scored_conditions = rescore_conditions(scored_conditions, group_shifts)
sync_figure_captures(scored_conditions)

print("\nRescored against group registrations:")
for cond in scored_conditions:
    print_capture(f"{cond['label']} [{cond['group_key']}]", cond["capture"])

describe_camera_grid(fig3_model, scored_conditions[0]["capture"].camera_shape, "Fig. 3 ")

### Verification

In [ ]:
"""
Verification before the figure cells.

1. The affine fold: a registration offset must collapse when folded in. A sign
   error doubles it instead, silently.
2. The camera grid: how the field of view lands on the capture, how much is
   scored once the zero order and erosion are removed.
3. The masks and the field-of-view outline, drawn over one capture.
"""
_probe = scored_conditions[0]

verify_shift_sign(_probe["model"], _probe["capture"], _probe["target_path"])
print()
describe_camera_grid(_probe["model"], _probe["capture"].camera_shape, "probe ")

_cap = _probe["capture"]
_model = _probe["model"]
_poly = fov_polygon_camera(_model, _cap.camera_shape)

fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH_MM / 25.4, FIG_WIDTH_MM / 75),
                         dpi=150)
for ax, (img, title) in zip(axes, [
        (_cap.measured, "measured (camera)"),
        (_cap.target, "target rendered to camera"),
        (_cap.mask.astype(float), "scored region")]):
    ax.imshow(img, cmap=CMAP_GRAY, interpolation="nearest")
    ax.plot(np.append(_poly[:, 0], _poly[0, 0]), np.append(_poly[:, 1], _poly[0, 1]),
            color=ACCENT_COLOR, lw=1)
    ax.set_title(title, fontsize=FONT_SIZE)
    ax.axis("off")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(FIG_WIDTH_MM / 50, FIG_WIDTH_MM / 50), dpi=150)
ax.imshow(crop_to_fov(_cap.measured, _model), cmap=CMAP_GRAY, interpolation="nearest")
ax.set_title("measured, cropped to the field of view", fontsize=FONT_SIZE)
ax.axis("off")
plt.show()

## Fig. 6 — stray light

In [ ]:
"""
Fig. 6, data - the recovered stray-light field B with and without the physical
SLM aperture, the far-field capture showing the cross artefact, and the
before/after captures for the computationally removed reflection.

background.get_filtered(mode="both") returns (E_masked, U_thres):
    U_thres  - B in its native far-field representation, threshold-limited
    E_masked - the same field taken back to the SLM plane, which is where the
                aperture edge and dust physically sit and therefore the only
                domain in which they are visible

The before and after captures come from different sessions, so each is scored
through its own registration rather than a shared one.

Slow (2 model loads, disk I/O, registration): run once.
"""
FIG6_CROSS_CAPTURE_DIR = CGH_DIR / "no_ap_4pi/tmgd_full_fov/batch_0/captures"
FIG6_BEFORE_DIR = CGH_DIR / "study_05_target_sweep/checkerboard_120"
FIG6_AFTER_DIR = (REPEAT_DIR / "study_06_module_ablation/background_on_repeat_5"
                  / "checkerboard_pitch_120")
FIG6_TARGET_PATH = FIG6_BEFORE_DIR / "target.png"
FIG6_BATCH_SIZE = 4
FIG6_N_BATCHES = 5

fig6_model_no_ap = HoloSystem.load(str(CKPT_4PI_NO_AP), device=DEVICE)
fig6_model_ap = HoloSystem.load(str(CKPT_4PI_AP), device=DEVICE)
assert_first_order(fig6_model_no_ap, "Fig. 6 (no aperture):")
assert_first_order(fig6_model_ap, "Fig. 6 (aperture):")

_ft_no_ap, _b_no_ap = fig6_model_no_ap.background.get_filtered(mode="both")
fig6_ft_no_ap = _ft_no_ap.detach().cpu().numpy()   # complex, SLM plane
fig6_b_no_ap = _b_no_ap.detach().cpu().numpy()     # complex, far field

_ft_ap, _b_ap = fig6_model_ap.background.get_filtered(mode="both")
fig6_ft_ap = _ft_ap.detach().cpu().numpy()
fig6_b_ap = _b_ap.detach().cpu().numpy()

# Cross-artefact capture, taken on the no-aperture system and therefore
# rectified through that system's own camera model.
fig6_cross_capture = rectify_to_far_field(
    load_capture_mean(FIG6_CROSS_CAPTURE_DIR, FIG6_BATCH_SIZE),
    fig6_model_no_ap, square=True)

fig6_mask = torch.load(str(BACKGROUND_MASK_PATH), map_location="cpu")
fig6_mask = np.asarray(fig6_mask.detach().cpu().numpy()
                       if torch.is_tensor(fig6_mask) else fig6_mask)
assert fig6_mask.ndim == 2, f"expected a 2-D mask, got shape {fig6_mask.shape}"
assert fig6_mask.shape == fig6_b_ap.shape, (
    f"mask {fig6_mask.shape} and B field {fig6_b_ap.shape} must share the native "
    "far-field grid - the figure cell crops both with the same region."
)

# Scored captures, so the removal can be quantified rather than only shown.
# Each session gets its own registration: the two were recorded months apart
# and there is no reason for their gauge offsets to agree.
fig6_before_cap = prepare_capture(FIG6_BEFORE_DIR, fig6_model_ap, FIG6_TARGET_PATH,
                                  FIG6_BATCH_SIZE, FIG6_N_BATCHES)
fig6_after_cap = prepare_capture(FIG6_AFTER_DIR, fig6_model_ap, FIG6_TARGET_PATH,
                                 FIG6_BATCH_SIZE, FIG6_N_BATCHES)

print("Prepared:")
print(f"  B at the SLM plane: no aperture {fig6_ft_no_ap.shape}, "
      f"aperture {fig6_ft_ap.shape}")
print(f"  cross capture: {fig6_cross_capture.shape}")
print(f"  mask: {fig6_mask.shape} (native far-field grid)")
print_capture("before", fig6_before_cap)
print_capture("after ", fig6_after_cap)

In [ ]:
"""
Fig. 6, figure - two rows.

Row 1 (a): the far-field capture with no aperture, showing the cross, with the
           recovered far-field B inset beside it.
      (b, c): amplitude and phase of that field taken back to the SLM plane,
           which is where the aperture edge and the dust producing the cross
           physically sit.
Row 2 (d, e): the far field with the aperture fitted, inset with the masked
           reflection selected for removal and marked where it was taken from.
      (f, g): that reflection before and after correction, with the metrics and
           the excess power beneath each.

The reflection sits beside the zero order, which the global mask excludes
because no hologram controls it, so (f, g)'s metrics are computed over the
whole region rather than through that mask.
"""
FIG6_REFLECTION_SPEC = (0.535, 0.495, 0.03)

FT_CROP_W, FT_CROP_H = 2340, 1740   # SLM-plane panels
A_INSET_CROP_SIZE = 3000            # far-field B inset in (a)
B_CROP_SIZE = 500                   # far-field crop for (d, e)

COL_GAP_MM = 2.0
ROW_GAP_MM = 12.0
FIG6_ROW1_MM = 50.0
FIG6_ROW2_MM = 40.0
FIG6_CBAR_MM = 9.0        # extra height for a panel's own colorbar and label
FIG6_METRICS_MM = 8.0
A_INSET_FRAC = 0.40
INSET_FRAC = 0.5
INSET_MARGIN = 0.0

# Amplitude carries the blue accent and phase the yellow, matching Fig. 2:
# the phase map is cyclic and mostly mid-tone, where blue reads poorly.
FIG6_AMP_COLOR = SECONDARY_COLOR
FIG6_PHASE_COLOR = ACCENT_COLOR


def far_field_region(model, fy: float, fx: float, size_frac: float) -> tuple:
    """
    Row/column slices of a physically square region on the far-field grid.

    The far field carries Mf x Nf samples over a square angular extent, so a
    square region is a rectangle in samples; drawn with a unit extent it
    renders square again.
    """
    Mf, Nf = model.geometry.far_fov_samples
    hy = max(int(round(size_frac * Mf)) // 2, 1)
    hx = max(int(round(size_frac * Nf)) // 2, 1)
    cy, cx = int(round(fy * Mf)), int(round(fx * Nf))
    return slice(cy - hy, cy + hy), slice(cx - hx, cx + hx)


_rows, _cols = far_field_region(fig6_model_ap, *FIG6_REFLECTION_SPEC)
roi_b_amp = np.abs(fig6_b_ap)[_rows, _cols]
roi_b_phase = np.angle(fig6_b_ap)[_rows, _cols]
roi_mask = fig6_mask[_rows, _cols]

def roi_for_capture(model, cap, spec):
    """
    The same physical region on this capture's own grid.

    Registration is folded into the target's rendering, not into the
    measurement, so two sessions with different gauge offsets put the same
    feature at different pixels. Offsetting the box by the capture's own
    registration puts it back on the feature. The target was rendered through
    that same offset, so one box serves both.
    """
    roi = roi_from_fov(model, cap.camera_shape, *spec)
    _, col_sl = fov_bbox(model, cap.camera_shape)
    px_per_far = (col_sl.stop - col_sl.start) / model.geometry.far_fov_samples[1]
    return ROI(row0=roi.row0 + int(round(cap.registration.dy * px_per_far)),
               col0=roi.col0 + int(round(cap.registration.dx * px_per_far)),
               size=roi.size)


fig6_before_roi = roi_for_capture(fig6_model_ap, fig6_before_cap, FIG6_REFLECTION_SPEC)
fig6_after_roi = roi_for_capture(fig6_model_ap, fig6_after_cap, FIG6_REFLECTION_SPEC)
fig6_cam_roi = fig6_before_roi          # scale bars and extents

roi_before = crop_roi(fig6_before_cap.measured, fig6_before_roi)
roi_after = crop_roi(fig6_after_cap.measured, fig6_after_roi)
target_before = crop_roi(fig6_before_cap.target, fig6_before_roi)
target_after = crop_roi(fig6_after_cap.target, fig6_after_roi)
metrics_before = compute_metrics(roi_before, target_before)
metrics_after = compute_metrics(roi_after, target_after)

print(f"  before ROI offset by {fig6_before_cap.registration}")
print(f"  after  ROI offset by {fig6_after_cap.registration}")

blob_mask = np.asarray(Image.fromarray(roi_mask.astype(np.uint8) * 255).resize(
    (roi_before.shape[1], roi_before.shape[0]), Image.NEAREST)) > 127


def blob_excess_power(crop: np.ndarray, target: np.ndarray) -> float:
    """
    Energy inside the masked blob that departs from the target.

    The region straddles a checkerboard edge, so a local background level is
    meaningless - it sits between the bright and dark squares and makes a blob
    on a dark square read as negative. The gain-matched target is the reference
    instead: what is summed is deviation from what was asked for, which is
    positive by construction and comparable between the two sessions.
    """
    return float(np.sum(np.abs(crop[blob_mask] - target[blob_mask])) / np.sum(blob_mask))


power_before = blob_excess_power(roi_before, target_before)
power_after = blob_excess_power(roi_after, target_after)

# Row 1 is [a | b | c] with a square; row 2 is four equal columns. The field
# panels are given the colorbar's height on top of the row, so the image part
# of (d, e) matches the full height of (f, g).
row1_wide = (FIG_WIDTH_MM - FIG6_ROW1_MM - 2 * COL_GAP_MM) / 2
row2_col = (FIG_WIDTH_MM - 3 * COL_GAP_MM) / 4
cols_row1 = axis_positions_mm([FIG6_ROW1_MM, row1_wide, row1_wide],
                              COL_GAP_MM, FIG_WIDTH_MM)
cols_row2 = axis_positions_mm([row2_col] * 4, COL_GAP_MM, FIG_WIDTH_MM)

v_sizes = [FIG6_ROW1_MM, FIG6_ROW2_MM, FIG6_METRICS_MM]
v_gaps = [ROW_GAP_MM, 1.0]
fig6_height_mm = sum(v_sizes) + sum(v_gaps)
v_pos = axis_positions_mm(v_sizes, v_gaps, fig6_height_mm)

fig6 = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, fig6_height_mm / 25.4))


def region_fraction_in_crop(model, spec: tuple, crop_w: int, crop_h: int) -> tuple:
    """
    Where a far-field region sits inside a centred crop, as axes fractions.

    The panels show a centre crop of the far field drawn with a unit extent, so
    the region's position has to be expressed relative to that crop rather than
    to the full grid.
    """
    fy, fx, size_frac = spec
    Mf, Nf = model.geometry.far_fov_samples
    x0 = (fx * Nf - (Nf - crop_w) / 2) / crop_w
    y0 = (fy * Mf - (Mf - crop_h) / 2) / crop_h
    return x0, 1.0 - y0, size_frac * Nf / crop_w, size_frac * Mf / crop_h


def draw_region_marker(ax, spec, crop_w, crop_h, color):
    """Outline on the panel showing where its inset was taken from."""
    xc, yc, w, h = region_fraction_in_crop(fig6_model_ap, spec, crop_w, crop_h)
    ax.add_patch(Rectangle((xc - w / 2, yc - h / 2), w, h, fill=False,
                           edgecolor=color, lw=1.0))


def complex_pair(cols_xw, v_index, field, letters, titles=(None, None),
                 crop_w=None, crop_h=None, insets=None, square=False,
                 inset_loc="lower left", scale_bars=None, extra_height_mm=0.0,
                 marker_spec=None):
    """
    Amplitude and phase of one complex field, each with a colorbar, its own
    letter and an optional inset.

    extra_height_mm extends the axes below the row so the colorbar does not eat
    into the image, which is what keeps these panels the same height as the
    square ones beside them.

    No set_box_aspect here: once the colorbar divider owns the axes, box aspect
    fights it for layout control.
    """
    amp, phase = np.abs(field), np.angle(field)
    if crop_w is not None:
        amp = center_crop(amp, crop_w, crop_h)
        phase = center_crop(phase, crop_w, crop_h)
    extent = dict(extent=(0, 1, 0, 1)) if square else {}
    colors = (FIG6_AMP_COLOR, FIG6_PHASE_COLOR)

    axes = []
    for (x0, w), data, cmap, label, kwargs in (
            (cols_xw[0], amp, CMAP_AMP, "Amplitude (a.u.)", {}),
            (cols_xw[1], phase, CMAP_PHASE, "Phase (rad)",
             dict(vmin=-np.pi, vmax=np.pi))):
        bottom, h = top_down_to_axes(*v_pos[v_index])
        extra = extra_height_mm / fig6_height_mm
        ax = fig6.add_axes([x0, bottom - extra, w, h + extra])
        ax.axis("off")
        im = ax.imshow(data, cmap=cmap, interpolation="nearest", **kwargs, **extent)
        add_horizontal_colorbar(fig6, ax, im, label, font_size=FONT_SIZE)
        axes.append(ax)

    if insets is not None:
        for k, (ax, (inset, cmap, kwargs, mask)) in enumerate(zip(axes, insets)):
            inset_ax = draw_image_inset(
                ax, inset, colors[k], cmap=cmap, width_frac=INSET_FRAC,
                height_frac=INSET_FRAC, margin_frac=INSET_MARGIN,
                corner=inset_loc, **kwargs)
            if mask is not None:
                overlay_binary_mask(inset_ax, mask, color="black")
            if marker_spec is not None:
                draw_region_marker(ax, marker_spec, crop_w or field.shape[1],
                                   crop_h or field.shape[0], colors[k])
            if scale_bars is not None and k == 0:
                draw_scale_bar_extent(ax, scale_bars[0])
                draw_scale_bar_extent(inset_ax, scale_bars[1])

    for ax, letter, title in zip(axes, letters, titles):
        draw_panel_letter(ax, letter, FONT_SIZE, title)
    return axes


# (a) the cross, with the recovered far-field B inset
_bottom, _h = top_down_to_axes(*v_pos[0])
x0, w = cols_row1[0]
ax_cross = fig6.add_axes([x0, _bottom, w, _h])
ax_cross.axis("off")
ax_cross.set_anchor("N")
ax_cross.set_box_aspect(1)
ax_cross.imshow(fig6_cross_capture, cmap=CMAP_GRAY, interpolation="nearest")
_amp_no_ap = np.abs(fig6_b_no_ap)
_ih = int(A_INSET_CROP_SIZE * _amp_no_ap.shape[0] / _amp_no_ap.shape[1])
draw_image_inset(ax_cross, center_crop(_amp_no_ap, A_INSET_CROP_SIZE, _ih),
                 ACCENT_COLOR, cmap=CMAP_AMP, width_frac=A_INSET_FRAC,
                 height_frac=A_INSET_FRAC, margin_frac=INSET_MARGIN,
                 corner="lower left")
draw_panel_letter(ax_cross, "a", FONT_SIZE, "No aperture")

# (b, c) that field at the SLM plane, no aperture
complex_pair(cols_row1[1:3], 0, fig6_ft_no_ap, ("b", "c"), crop_w=FT_CROP_W,
             crop_h=FT_CROP_H)

# (d, e) the far field with the aperture fitted, inset with the reflection and
# marked where it came from
_bh = int(B_CROP_SIZE * fig6_b_ap.shape[0] / fig6_b_ap.shape[1])
_fov_um = fov_extent_um(fig6_model_ap, fig6_before_cap.camera_shape)
complex_pair(cols_row2[0:2], 1, fig6_b_ap, ("d", "e"),
             crop_w=B_CROP_SIZE, crop_h=_bh, square=True,
             insets=[(roi_b_amp, CMAP_AMP, {}, roi_mask),
                     (roi_b_phase, CMAP_PHASE, dict(vmin=-np.pi, vmax=np.pi), roi_mask)],
             inset_loc="upper right", marker_spec=FIG6_REFLECTION_SPEC,
             extra_height_mm=FIG6_CBAR_MM,
             scale_bars=(_fov_um * B_CROP_SIZE / fig6_b_ap.shape[1],
                         _fov_um * FIG6_REFLECTION_SPEC[2]))

# (f, g) the reflection before and after, on one intensity scale
_r2_bottom, _r2_h = top_down_to_axes(*v_pos[1])
_txt_bottom, _txt_h = top_down_to_axes(*v_pos[2])
for (x0, w), crop, metrics, power, letter, title in (
        (cols_row2[2], roi_before, metrics_before, power_before, "f", "Before"),
        (cols_row2[3], roi_after, metrics_after, power_after, "g", "After")):
    ax = fig6.add_axes([x0, _r2_bottom, w, _r2_h])
    ax.axis("off")
    ax.set_box_aspect(1)
    ax.imshow(crop, cmap=CMAP_GRAY, vmin=0, vmax=1, interpolation="nearest")
    # The mask boundary as a contour rather than a tinted overlay: it stays a
    # vector path in the PDF, and it does not alter the pixels underneath it.
    ax.contour(blob_mask.astype(float), levels=[0.5], colors=SECONDARY_COLOR,
               linewidths=1.5, antialiased=True)
    draw_panel_letter(ax, letter, FONT_SIZE, title)
    if letter == "f":
        draw_scale_bar_extent(ax, roi_extent_um(fig6_cam_roi, fig6_model_ap), corner="lower left")
    ax_txt = fig6.add_axes([x0, _txt_bottom, w, _txt_h])
    ax_txt.axis("off")
    ax_txt.text(0.5, 1.0,
                f"NMSE {metrics['NMSE']:.3f}, {metrics['PSNR']:.1f} dB\n"
                f"excess power {power:.3g}",
                transform=ax_txt.transAxes, fontsize=FONT_SIZE - 1,
                ha="center", va="top", linespacing=1.4)

save_figure(fig6, "fig6_stray_light")

## Fig. 5 — camera model and sampling

In [ ]:
"""
Fig. 5, data - twin predictions against real captures, on the camera's own
pixel grid.
 
Every other figure rectifies captures onto the far-field grid. This one does
not: the artefacts it is about belong to the observation pipeline and would be
warped away. The target is instead put through the same trained camera, so
every panel shows what a camera would see rather than an underlying field.
 
The per-frame scores come from the same targets as Fig. 3, each hologram
predicted and scored against the frame it produced, so the twin is tested
twenty times per target rather than once on an average.
 
Slow (one CGH forward pass per frame): run once. Reads fig3_captures, so run
Fig. 3 first.
"""
FIG5_CHECKPOINT = CKPT_4PI_AP
FIG5_REGISTRATION = fig3_captures["complex"].registration
 
FIG5_SINGLE_DIR = CGH_DIR / "study_05_target_sweep/complex_single"
FIG5_COMPLEX_DIR = CGH_DIR / "study_05_target_sweep/complex"
FIG5_BATCH_SIZE = 4
FIG5_N_BATCHES = 5
 
PER_FRAME_TARGETS = FIG3_TARGETS
PER_FRAME_BASE_DIR = FIG3_BASE_DIR
 
fig5_model = HoloSystem.load(str(FIG5_CHECKPOINT), device=DEVICE)
 
 
# Camera-space panel cropped to the field of view, without resampling.
def crop_to_fov_view(img):
    row_sl, col_sl = fov_bbox(fig5_model, img.shape[:2])
    return img[row_sl, col_sl]
 
 
# --- image panels ----------------------------------------------------------
 
fig5_capture_single = load_capture_mean(FIG5_SINGLE_DIR, batch_size=1, n_batches=1)
fig5_capture_single = fig5_capture_single / fig5_capture_single.max()
 
 
fig5_intensity_single, fig5_predicted_single = predict_camera(
    fig5_model, load_holograms(FIG5_SINGLE_DIR / "batch_0"),
    fig5_capture_single.shape, FIG5_REGISTRATION)
 
with registered_camera(fig5_model, FIG5_REGISTRATION):
    fig5_target_far = load_target_image(FIG5_COMPLEX_DIR / "target.png",
                                        fig5_model.geometry.far_fov_samples)
    fig5_target = render_to_camera(fig5_model, fig5_target_far,
                                   fig5_capture_single.shape)
fig5_target = fig5_target / fig5_target.max()
 
fig5_capture_avg = load_capture_mean(FIG5_COMPLEX_DIR, FIG5_BATCH_SIZE, FIG5_N_BATCHES)
fig5_capture_avg = fig5_capture_avg / fig5_capture_avg.max()
_, fig5_predicted_avg = predict_camera(
    fig5_model, load_holograms(FIG5_COMPLEX_DIR, FIG5_N_BATCHES),
    fig5_capture_avg.shape, FIG5_REGISTRATION)
 
fig5_target = crop_to_fov_view(fig5_target)
fig5_view_single = crop_to_fov_view(fig5_capture_single)
fig5_valid = crop_to_fov_view(metric_mask(fig5_model, fig5_capture_single.shape))
 
print("Image panels:")
print(f"  target {fig5_target.shape}, measured {fig5_view_single.shape}")
 
# --- per-frame scores ------------------------------------------------------
 
# Every capture frame for one condition, in the order the holograms were shown.
def load_frames(base_dir, batch_size=FIG5_BATCH_SIZE, n_batches=FIG5_N_BATCHES):
    return [np.load(Path(base_dir, f"batch_{n}", "captures", f"{k}.npy"))
            for n in range(n_batches) for k in range(batch_size)]
 
 
# Predicted camera image for a single hologram, through the corrected affine.
def predict_frame(model, hologram, camera_shape, registration, exposure_time):
    _, predicted = predict_camera(model, hologram[None], camera_shape,
                                  registration, exposure_time)
    return predicted / predicted.max()
 
 
# Score every frame of one target, and its time average.
def score_frames(name):
    capture = fig3_captures[name]
    base = PER_FRAME_BASE_DIR / name
    holograms = load_holograms(base, FIG6_N_BATCHES)
    frames = load_frames(base)
    exposure = exposure_for(name)
    target, mask = capture.target, capture.mask
 
    rows = []
    for hologram, frame in zip(holograms, frames):
        measured = frame / frame.max()
        predicted = predict_frame(fig3_model, hologram, capture.camera_shape,
                                  capture.registration, exposure)
        rows.append(dict(
            meas_target=compute_metrics(measured, target, mask),
            pred_target=compute_metrics(predicted, target, mask),
            pred_meas=compute_metrics(measured, predicted, mask)))
 
    _, predicted_mean = predict_camera(fig3_model, holograms, capture.camera_shape,
                                       capture.registration, exposure)
    predicted_mean = predicted_mean / predicted_mean.max()
    averaged = dict(
        meas_target=capture.metrics,
        pred_target=compute_metrics(predicted_mean, target, mask),
        pred_meas=compute_metrics(capture.measured, predicted_mean, mask))
    return dict(name=name, frames=rows, averaged=averaged)
 
 
per_frame = {name: score_frames(name) for name in PER_FRAME_TARGETS}
 
print("\nPer-frame NMSE (median over frames) against the time average:")
for name, record in per_frame.items():
    frame_mt = np.array([r["meas_target"]["NMSE"] for r in record["frames"]])
    print(f"  {name:<18s} M-T {np.median(frame_mt):.4f} per frame -> "
          f"{record['averaged']['meas_target']['NMSE']:.4f} averaged")

In [ ]:
"""
Fig. 5, figure - six matched panels above, then the line scans and the
per-frame statistics.

Row 1 (a-f): the region across the target, the twin and the measurement.
Row 2 (g): the same cut through all six, stacked so each trace has its own
           band instead of six curves sharing one axis.
      (h, i): every frame of every target scored against its own prediction,
           in PSNR and in SSIM.

The three panels of row 2 share an axes height, so the statistics panels are
given whatever width their square aspect needs and the line scans take the
rest - sizing the slots first would leave the squares shrunk inside them.
"""
FIG5_ROI_SPEC = (0.592, 0.648, 0.03)
FIG5_ROI = roi_from_fov(fig5_model, fig5_view_single.shape, *FIG5_ROI_SPEC)

COL_GAP_MM = 2.0
ROW_GAP_MM = 2.0
FIG5_LETTER_MM = 4.0             # label band above the images
FIG5_TITLE_X = 0.11              # title offset from the letter, in axes fraction
FIG5_METRICS_GAP_MM = 2.0        # between an image and its metrics
FIG5_METRICS_MM = 7.0

FIG5_ROW2_MM = 54.0
COL_GAP_MM_ROW_2 = 10
FIG5_ROW2_MARGIN_MM = (0, 0, 1, 0.0)     # left, bottom, top, right

PER_FRAME_COLORS = {
    "complex": ACCENT_COLOR, "checkerboard_120": SECONDARY_COLOR,
    "checkerboard_10": "red", "meta": "#7B2CBF",
    "grayscale": "#17C42E", "usaf": "black",
}

PER_FRAME_NAMES = {
    "complex": "Complex", "checkerboard_120": "Chequerboard (120)",
    "checkerboard_10": "Chequerboard (10)", "meta": "Huygens pillars",
    "grayscale": "Greyscale", "usaf": "USAF",
}

TRACE_LOCATION = 0.17
TRACE_REFERENCE_COLOR = SECONDARY_COLOR
TRACE_COLOR = "black"

fig5_col_mm = (FIG_WIDTH_MM - 5 * COL_GAP_MM) / 6
row1_mm = FIG5_LETTER_MM + fig5_col_mm + FIG5_METRICS_GAP_MM + FIG5_METRICS_MM
fig5_height_mm = row1_mm + ROW_GAP_MM + FIG5_ROW2_MM

# The statistics panels are square, so their slot is set by the shared axes
# height; the line scans take whatever is left.
_left, _bottom, _top, _right = FIG5_ROW2_MARGIN_MM
row2_axes_mm = FIG5_ROW2_MM - _top - _bottom
stat_slot_mm = row2_axes_mm + _left + _right
profile_slot_mm = FIG_WIDTH_MM - 2 * stat_slot_mm - 2 * COL_GAP_MM_ROW_2

fig5 = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, fig5_height_mm / 25.4))
fig5_cols = axis_positions_mm([fig5_col_mm] * 6, COL_GAP_MM, FIG_WIDTH_MM)


# The shared region on this image's own grid, at its native resolution.
def fig5_region(image):
    row_sl, col_sl = roi_in_other_grid(FIG5_ROI, fig5_view_single.shape, image.shape)
    return image[row_sl, col_sl]


# Predicted against measured, full field and region, on the scored mask.
def fig5_pair_metrics(predicted, measured):
    full = compute_metrics(measured, predicted, fig5_valid)
    region = compute_metrics(fig5_region(measured), fig5_region(predicted),
                             fig5_region(fig5_valid))
    return format_full_and_roi(full, region)


# Letter and name above a panel, at one size throughout the figure.
def fig5_label(ax, letter, title):
    ax.text(0, 0.1, letter, transform=ax.transAxes, fontsize=FONT_SIZE+1,
            fontweight="bold", ha="left", va="bottom")
    ax.text(FIG5_TITLE_X, 0.1, title, transform=ax.transAxes, fontsize=FONT_SIZE,
            ha="left", va="bottom")


# One region panel, with its label above the image rather than over it.
def fig5_panel(col, image, letter, title, scale_bar=False, line_at=None):
    x0, w = fig5_cols[col]
    ax_label = fig5.add_axes([x0, 1 - FIG5_LETTER_MM / fig5_height_mm, w,
                              FIG5_LETTER_MM / fig5_height_mm])
    ax_label.axis("off")
    fig5_label(ax_label, letter, title)

    ax = fig5.add_axes([x0, 1 - (FIG5_LETTER_MM + fig5_col_mm) / fig5_height_mm,
                        w, fig5_col_mm / fig5_height_mm])
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(fig5_region(image), cmap=CMAP_GRAY, interpolation="nearest")
    ax.set_box_aspect(1)
    if line_at is not None:
        ax.axhline(line_at * fig5_region(image).shape[0],
                   color=TRACE_REFERENCE_COLOR, lw=2.0)
    if scale_bar:
        draw_scale_bar_extent(ax, roi_extent_um(FIG5_ROI, fig5_model),
                              corner="lower left")
    return ax


PANELS = [
    (fig5_target, "a", "Target"),
    (np.roll(fig5_intensity_single, shift=(-2, -6), axis=(0, 1)), "b", "Predicted intensity"),
    (crop_to_fov_view(fig5_predicted_single), "c", "Predicted frame"),
    (fig5_view_single, "d", "Measured frame"),
    (crop_to_fov_view(fig5_predicted_avg), "e", "Predicted average"),
    (crop_to_fov_view(fig5_capture_avg), "f", "Measured average"),
]

panel_axes = [fig5_panel(col, image, letter, title, scale_bar=(col == 0),
                         line_at=TRACE_LOCATION if col == 0 else None)
              for col, (image, letter, title) in enumerate(PANELS)]

# Metrics beneath the two measured panels, each against its own prediction.
for ax, predicted, measured in (
        (panel_axes[3], crop_to_fov_view(fig5_predicted_single), fig5_view_single),
        (panel_axes[5], crop_to_fov_view(fig5_predicted_avg),
         crop_to_fov_view(fig5_capture_avg))):
    box = ax.get_position()
    height = FIG5_METRICS_MM / fig5_height_mm
    gap = FIG5_METRICS_GAP_MM / fig5_height_mm
    ax_txt = fig5.add_axes([box.x0, box.y0 - gap - height, box.width, height])
    ax_txt.axis("off")
    ax_txt.text(0.5, 1.0, fig5_pair_metrics(predicted, measured),
                transform=ax_txt.transAxes, fontsize=FONT_SIZE - 2,
                ha="center", va="top", linespacing=1.4)


# Axes in the second row, at the shared height.
def fig5_row2_axes(x0_mm, width_mm):
    left, bottom, top, right = FIG5_ROW2_MARGIN_MM
    return fig5.add_axes([
        (x0_mm + left) / FIG_WIDTH_MM,
        bottom / fig5_height_mm,
        (width_mm - left - right) / FIG_WIDTH_MM,
        row2_axes_mm / fig5_height_mm,
    ])


# --- (g) line scans, separate subplots stacked vertically -------------------
left, bottom, top, right = FIG5_ROW2_MARGIN_MM
num_traces = len(PANELS)
sub_h_mm = row2_axes_mm / num_traces

for index, (image, letter, title) in enumerate(PANELS):
    y_bottom_mm = bottom + (num_traces - 1 - index) * sub_h_mm
    ax = fig5.add_axes([
        (0.0 + left) / FIG_WIDTH_MM,
        y_bottom_mm / fig5_height_mm,
        (profile_slot_mm - left - right) / FIG_WIDTH_MM,
        sub_h_mm / fig5_height_mm,
    ])
    
    crop = fig5_region(image)
    mid = int(crop.shape[0] * TRACE_LOCATION)
    trace = crop[mid]
    color = TRACE_REFERENCE_COLOR if index == 0 else TRACE_COLOR
    
    ax.plot(np.linspace(0, 1, trace.size), trace / trace.mean(),
            color=color, lw=0.9)
    # ax.text(0.97, 0.82, f"{letter}  {title}", transform=ax.transAxes,
    #         fontsize=FONT_SIZE - 2, ha="right", va="top")
    ax.text(0.01, 0.9, f"{letter}", transform=ax.transAxes, fontweight="bold",
            fontsize=FONT_SIZE, ha="left", va="top")

    ax.set_yticks([])
    if index == 2:
        ax.set_ylabel(f"Relative brightness", fontsize=FONT_SIZE - 1)
        ax.yaxis.set_label_coords(-0.03, -0.05) 
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=FONT_SIZE - 2, length=2)
    ax.grid(True, color="0.85", lw=0.5, axis="x")
    ax.set_axisbelow(True)
    
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(0.8)

    if index < num_traces - 1:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Position across the region", fontsize=FONT_SIZE - 1)

    if index == 0:
        axes_panel_label(ax, "g", "Line scans", font_size=FONT_SIZE)


# --- (h, i) image statistics -------------------------------------------
#
# One point per frame, a star per time average
def fig5_statistics(slot, metric, letter, legend=False):
    x0 = profile_slot_mm + COL_GAP_MM + COL_GAP_MM_ROW_2 + slot * (stat_slot_mm + COL_GAP_MM_ROW_2)
    ax = fig5_row2_axes(x0, stat_slot_mm)

    for name, record in per_frame.items():
        color = PER_FRAME_COLORS[name]
        label = PER_FRAME_NAMES[name]
        ax.scatter([r["meas_target"][metric] for r in record["frames"]],
                   [r["pred_target"][metric] for r in record["frames"]],
                   s=7, color=color, alpha=0.65, lw=0, label=label, zorder=3)
        ax.scatter(record["averaged"]["meas_target"][metric],
                   record["averaged"]["pred_target"][metric],
                   s=45, color=color, marker="*", lw=0.4, edgecolor="white",
                   zorder=4)

    values = np.array([v for record in per_frame.values() for r in record["frames"]
                       for v in (r["meas_target"][metric], r["pred_target"][metric])])
    values = np.append(values, [(record["averaged"]["meas_target"][metric],
                                 record["averaged"]["pred_target"][metric])
                                for record in per_frame.values()])
    low, high = (0.0, 1.0) if metric == "SSIM" else (values.min(), values.max())
    pad = 0.04 * (high - low)
    low, high = low - pad, high + pad
    ax.plot([low, high], [low, high], color="0.6", ls="--", lw=0.8, zorder=1)

    ax.set_xlim(low, high); ax.set_ylim(low, high)
    ax.set_box_aspect(1)
    metric_label = "SSIM" if metric == "SSIM" else f"{metric} (dB)"
    ax.set_xlabel(f"Measured vs target, {metric_label}", fontsize=FONT_SIZE - 1)
    ax.set_ylabel(f"Predicted vs target, {metric_label}", fontsize=FONT_SIZE - 1)
    ax.tick_params(labelsize=FONT_SIZE - 2, length=2)
    ax.grid(True, color="0.92", lw=0.5)
    ax.set_axisbelow(True)
    if legend:
        ax.legend(loc="lower right", frameon=False, fontsize=FONT_SIZE - 1,
                  handletextpad=0.1, borderaxespad=0.2, labelspacing=0.2,
                  title="Dots: frames, Stars: averages",
                  title_fontsize=FONT_SIZE - 1)
    axes_panel_label(ax, letter, metric, font_size=FONT_SIZE)
    return ax


fig5_statistics(0, "PSNR", "h", legend=True)
fig5_statistics(1, "SSIM", "i")

save_figure(fig5, "fig5_camera_model")

Supplementary figure - probes, masks and fits at different stages

In [ ]:
"""
Supplementary figure - what the twin sees at each training stage.

Top row: the measurement the stage was fitted against. Bottom row: the twin's
own render of the same hologram once that stage had converged. Left to right,
the stages add what they model - alignment, then the optics, then the stray
light - so the pair in each column is the residual that stage was left with.

The insets carry the two masks and the probe capture they were built from, at
the same physical region in all three, so the region excluded from the loss can
be seen next to the images it was excluded from.
"""
import cv2

STAGE_DIR = FIT_DIR / "study_04_4pi_ap_fit/ap_4pi"
PROBE_CAPTURE = PAPER_ROOT / "paper_training_data/ap_4pi/mask/captures/imgs/0.png"

STAGE_COLUMNS = [
    ("camera_stage", "Alignment", STAGE_DIR / "camera_stage"),
    ("optics_stage", "Optics", STAGE_DIR / "optics_stage"),
    ("background_stage", "Stray light", STAGE_DIR / "background_stage"),
]

STAGE_INSETS = [
    ("Probe", PROBE_CAPTURE),
    ("Zero-order mask", STAGE_DIR / "ZOD Mask.png"),
    ("Saturation mask", STAGE_DIR / "Saturation Mask.png"),
]

INSET_CROP_PX = 200          # on the mask images' grid
INSET_FRAC = 0.34
INSET_MARGIN = 0.03
COL_GAP_MM, ROW_GAP_MM, LETTER_MM = 2.0, 2.0, 4.0


# Grayscale image as a float array in [0, 1].
def load_png(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img.astype(np.float64) / 255.0


# The same physical region from every image, whatever its sampling. The probe
# capture is on a different grid to the masks but shares their centre, so the
# crop is scaled by its width rather than taken at a fixed pixel count.
def central_crop(img, crop_px=INSET_CROP_PX):
    cy, cx = img.shape[0] // 2, img.shape[1] // 2
    half = max(crop_px // 2, 1)
    offset_y, offset_x = -5, 20
    return img[cy - half + offset_y:cy + half + offset_y, cx - half + offset_x:cx + half + offset_x]


stage_images = {}
for key, _, folder in STAGE_COLUMNS:
    stage_images[key] = (load_png(folder / "sample_camera_true.png"),
                         load_png(folder / "sample_camera_pred.png"))

inset_images = [(label, load_png(path)) for label, path in STAGE_INSETS]
reference_width = inset_images[1][1].shape[1]      # the masks set the scale
insets = [(label, central_crop(img)) for label, img in inset_images]

print("Loaded:")
for (key, name, _), (true, pred) in zip(STAGE_COLUMNS, stage_images.values()):
    print(f"  {name:<12s} measured {true.shape}, predicted {pred.shape}")
for label, crop in insets:
    print(f"  {label:<16s} crop {crop.shape}")

# --- figure ----------------------------------------------------------------
panel_mm = (FIG_WIDTH_MM - 2 * COL_GAP_MM) / 3
height_mm = 2 * (LETTER_MM + panel_mm) + ROW_GAP_MM
fig = plt.figure(figsize=(FIG_WIDTH_MM / 25.4, height_mm / 25.4))
cols = axis_positions_mm([panel_mm] * 3, COL_GAP_MM, FIG_WIDTH_MM)


# One image panel, optionally with a corner inset.
def stage_panel(col, row, image, letter, title, inset=None):
    x0, w = cols[col]
    top_mm = row * (LETTER_MM + panel_mm + ROW_GAP_MM)

    ax_letter = fig.add_axes([x0, 1 - (top_mm + LETTER_MM) / height_mm, w,
                              LETTER_MM / height_mm])
    ax_letter.axis("off")
    ax_letter.text(0, 0, letter, transform=ax_letter.transAxes,
                   fontsize=FONT_SIZE + 1, fontweight="bold", ha="left", va="bottom")
    ax_letter.text(0, 0, "    " + title, transform=ax_letter.transAxes,
                   fontsize=FONT_SIZE, ha="left", va="bottom")

    ax = fig.add_axes([x0, 1 - (top_mm + LETTER_MM + panel_mm) / height_mm, w,
                       panel_mm / height_mm])
    ax.axis("off")
    ax.imshow(image, cmap=CMAP_GRAY, interpolation="nearest")
    ax.set_box_aspect(1)
    if inset is not None:
        label, crop = inset
        inset_ax = draw_image_inset(ax, crop, ACCENT_COLOR, cmap=CMAP_GRAY,
                                    width_frac=INSET_FRAC, height_frac=INSET_FRAC,
                                    margin_frac=INSET_MARGIN, corner="lower right")
    return ax


for col, ((key, name, _), inset) in enumerate(zip(STAGE_COLUMNS, insets)):
    measured, predicted = stage_images[key]
    stage_panel(col, 0, measured, "abc"[col], f"{name}, measured", inset=inset)
    stage_panel(col, 1, predicted, "def"[col], f"{name}, predicted")

save_figure(fig, "figS_training_stages", pad_inches=0.02)